# Figure reproduction

This notebook contains the plotting/chart code separated from the main
computational analysis. It expects the corresponding result CSV files to exist
in `results/`.



## Descriptive statistics of WQ (Table 1)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

outdir = Path("analysis_outputs")
outdir.mkdir(exist_ok=True)

parameters = ["Turbidity", "EC", "DO", "Temp"]  # adjust if you use "Temp"

storm_stats_records = []

for storm_id, storm_info in all_storms.items():
    df = storm_info["data"].copy()

    for parameter in parameters:
        if parameter not in df.columns:
            print(f"Skipping {parameter} for Storm {storm_id}: column not found.")
            continue

        x = pd.to_numeric(df[parameter], errors="coerce").dropna()

        if len(x) == 0:
            continue

        storm_stats_records.append({
            "storm_id": storm_id,
            "parameter": parameter,
            "n": len(x),
            "mean": x.mean(),
            "median": x.median(),
            "std": x.std(),
            "min": x.min(),
            "max": x.max(),
            "range": x.max() - x.min(),
            "q25": x.quantile(0.25),
            "q75": x.quantile(0.75),
            "iqr": x.quantile(0.75) - x.quantile(0.25),
            "cv_percent": 100 * x.std() / x.mean() if x.mean() != 0 else np.nan,
        })

storm_stats = pd.DataFrame(storm_stats_records)

storm_stats.to_csv(outdir / "storm_wq_basic_statistics_long.csv", index=False)

display(storm_stats)

## Reference storm-event signals (Figure 1)


In [ ]:
# Source development-notebook cell 10
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
from pathlib import Path

# ----------------------------------------------------
# Settings
# ----------------------------------------------------
parameters = ["Turbidity", "EC", "DO", "Temp"]
storm_ids  = [1, 2, 3, 4]

param_colors = {
    "Turbidity": "#2166ac",
    "EC":        "#4dac26",
    "DO":        "#d01c8b",
    "Temp":      "#f4a582",
}

param_units = {
    "Turbidity": "NTU",
    "EC":        "µS/cm",
    "DO":        "%",
    "Temp":      "°C",
}

param_labels = {
    "Turbidity": "Turbidity",
    "EC":        "Electrical conductivity",
    "DO":        "Dissolved oxygen",
    "Temp":      "Water temperature",
}

output_dir = Path("figure_outputs")
output_dir.mkdir(exist_ok=True)

# ----------------------------------------------------
# Figure style
# ----------------------------------------------------
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": 12,
    "axes.titlesize": 12,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.titlesize": 15,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white"
})

# ----------------------------------------------------
# Helper functions
# ----------------------------------------------------
def get_signal_values(all_storms, storm_id, parameter):
    """
    Return a 1D numpy array for a storm parameter.
    """

    signal = get_reference_series(
        all_storms=all_storms,
        storm_id=storm_id,
        parameter=parameter,
    )

    values = signal.values if hasattr(signal, "values") else np.asarray(signal)
    values = np.asarray(values, dtype=float)

    return values


def clean_invert_normalize_radar(
    all_storms,
    storm_id,
    radar_parameter="Radar",
    low_atol=1.0,
    high_threshold=5.0,
    interpolate=True
):
    """
    Clean, invert, and normalize radar-derived stage.

    Cleaning:
    - Converts to numeric
    - Removes near-zero artifacts
    - Removes unrealistically high values
    - Optionally interpolates short gaps

    Inversion:
    - Uses max(cleaned radar) - radar so storm response rises upward

    Normalization:
    - Scales from 0 to 1 for plotting on secondary axis
    """

    radar = get_reference_series(
        all_storms=all_storms,
        storm_id=storm_id,
        parameter=radar_parameter,
    )

    radar = pd.Series(radar).copy()
    radar = pd.to_numeric(radar, errors="coerce")

    # Remove radar artifacts
    radar[(np.isclose(radar, 0, atol=low_atol)) | (radar > high_threshold)] = np.nan

    # Optional: interpolate gaps caused by cleaning
    if interpolate:
        radar = radar.interpolate(limit_direction="both")

    values = radar.to_numpy(dtype=float)

    finite = values[np.isfinite(values)]

    if len(finite) == 0:
        return values

    # Invert radar so larger plotted values indicate higher stage response
    radar_inv = np.nanmax(finite) - values

    # Normalize to 0–1 for secondary axis
    finite_inv = radar_inv[np.isfinite(radar_inv)]

    if len(finite_inv) == 0 or np.nanmax(finite_inv) == np.nanmin(finite_inv):
        return radar_inv

    radar_norm = (
        (radar_inv - np.nanmin(finite_inv)) /
        (np.nanmax(finite_inv) - np.nanmin(finite_inv))
    )

    return radar_norm


def robust_ylim(values, pad_fraction=0.08):
    """
    Create y-limits using finite values and padding.
    """

    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        return None

    ymin = np.nanmin(values)
    ymax = np.nanmax(values)

    yrange = ymax - ymin

    if yrange == 0:
        pad = 1
    else:
        pad = pad_fraction * yrange

    return ymin - pad, ymax + pad


In [ ]:
# Source development-notebook cell 11
# ----------------------------------------------------
# Create figure
# ----------------------------------------------------
n_storms = len(storm_ids)
n_params = len(parameters)

fig = plt.figure(figsize=(14, 8.8)) #4.8 * n_storms, 2.8 * n_params

gs = gridspec.GridSpec(
    n_params,
    n_storms,
    figure=fig,
    hspace=0.18, #0.32
    wspace=0.16 #0.28
)

plt.subplots_adjust(
    left=0.07,
    right=0.97,
    top=0.90,
    bottom=0.08
)

for row, parameter in enumerate(parameters):

    for col, storm_id in enumerate(storm_ids):

        ax = fig.add_subplot(gs[row, col])

        values = get_signal_values(
            all_storms=all_storms,
            storm_id=storm_id,
            parameter=parameter
        )

        x = np.arange(len(values))

        ax.plot(
            x,
            values,
            color=param_colors[parameter],
            linewidth=1.5,
            alpha=0.95
        )

        # ------------------------------------------------
        # Add cleaned, inverted, normalized radar only to turbidity row
        # ------------------------------------------------
        if parameter == "Turbidity":

            radar_norm = clean_invert_normalize_radar(
                all_storms=all_storms,
                storm_id=storm_id,
                radar_parameter="Radar",
                low_atol=1.0,
                high_threshold=5.0,
                interpolate=True
            )

            ax2 = ax.twinx()
            
            radar_color = "red"    
            ax2.plot(
                np.arange(len(radar_norm)),
                radar_norm,
                color=radar_color,
                linewidth=1.2,
                linestyle="--",
                alpha=0.65
            )

            ax2.set_ylim(-0.05, 1.2)

            if col == n_storms - 1:
                ax2.set_ylabel(
                    "Water level (m)",
                    fontsize=13,
                    fontweight="normal",
                    color=radar_color
                )
                ax2.tick_params(axis="y", labelsize=9, colors=radar_color)
            else:
                ax2.set_yticklabels([])
                ax2.tick_params(axis="y", length=0)

            ax2.spines["top"].set_visible(False)

        # Titles only in first row
        if row == 0:
            ax.set_title(
                f"Event {storm_id}",
                fontsize=12,
                fontweight="bold",
                pad=6
            )

        # Y-label only first column
        if col == 0:
            ax.set_ylabel(
                f"{param_labels[parameter]}/n({param_units[parameter]})",
                fontsize=13,
                fontweight="bold"
            )

        # X-label only bottom row
        if row == n_params - 1:
            ax.set_xlabel("Time step", fontsize=16)
        else:
            ax.set_xticklabels([])

        # Adaptive y-limit per subplot for readability
        ylim = robust_ylim(values)
        if ylim is not None:
            ax.set_ylim(ylim)

        ax.tick_params(axis="both", labelsize=13, length=3)
        ax.locator_params(axis="x", nbins=4)
        ax.locator_params(axis="y", nbins=4)

        ax.spines[["top", "right"]].set_visible(False)
        ax.grid(True, alpha=0.20, linewidth=0.6)

#fig.suptitle(
    #"Reference Water-Quality Signals for Selected Storm Events",
    #fontsize=15,
    #fontweight="bold",
    #y=0.995
#)

#fig.tight_layout(rect=[0, 0, 1, 0.975])

png_path = output_dir / "Figure_1_reference_signals_radar_secondary_adaptive_ylim.png"
pdf_path = output_dir / "Figure_1_reference_signals_radar_secondary_adaptive_ylim.pdf"
svg_path = output_dir / "Figure_1_reference_signals_radar_secondary_adaptive_ylim.svg"

fig.savefig(png_path, dpi=600, bbox_inches="tight")
fig.savefig(pdf_path, bbox_inches="tight")
fig.savefig(svg_path, bbox_inches="tight")

plt.show()

print("Saved:")
print(png_path)
print(pdf_path)
print(svg_path)


## Practical Accuracy Rating (Figure 2)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ----------------------------------------------------
# Load practical USGS rating summary
# ----------------------------------------------------
df = pd.read_csv("practical_usgs_rating_summary.csv")


# ----------------------------------------------------
# Short display names for algorithms
# ----------------------------------------------------
algo_display = {
    "Linear_Interpolation": "Linear",
    "Linear Interpolation": "Linear",

    "Cubic_Spline": "Spline",
    "Cubic Spline": "Spline",
    "Cubic_Spline_Interpolation": "Spline",
    "Cubic Spline Interpolation": "Spline",

    "PCHIP": "PCHIP",
    "Local_Moving_Mean": "Moving mean",
    "Delta_H_Weighted_Interpolation": "ΔH weighted",
    "HPWI": "HPWI",

    "Kalman_Filter": "Kalman",
    "Kalman Filter": "Kalman",
    "Local_Level_Kalman_Smoother": "Kalman",
    "Local-Level Kalman Smoother": "Kalman",

    "SSA_Gap_Filling": "SSA gap",
    "SSA Gap Filling": "SSA gap",

    "Median_Filter": "Median",
    "Median Filter": "Median",

    "Savitzky_Golay": "S-G",
    "Savitzky-Golay": "S-G",
    "Savitzky–Golay": "S-G",

    "Wavelet_Filter": "Wavelet",
    "Wavelet Filter": "Wavelet",
    "Wavelet": "Wavelet",

    "CEEMDAN": "CEEMDAN",
    "LOESS": "LOESS",
    "EWMA": "EWMA",
    "VMD": "VMD",
    "SSA": "SSA"
}


# ----------------------------------------------------
# Parameter display names
# ----------------------------------------------------
parameter_display = {
    "DO": "DO",
    "EC": "EC",
    "Temp": "Temperature",
    "Turbidity": "Turbidity"
}


# ----------------------------------------------------
# Basic cleaning
# ----------------------------------------------------
df["scenario"] = df["scenario"].astype(str)
df["parameter"] = df["parameter"].astype(str)
df["algorithm"] = df["algorithm"].astype(str)
df["evaluation_domain"] = df["evaluation_domain"].astype(str)


# ----------------------------------------------------
# Create rating components
# ----------------------------------------------------
df["Excellent"] = df["Excellent_rate_mean"]

df["Good_only"] = (
    df["Good_or_better_rate_mean"]
    - df["Excellent_rate_mean"]
)

df["Poor"] = df["Poor_rate_mean"]

df["Fair_only"] = (
    100
    - df["Good_or_better_rate_mean"]
    - df["Poor_rate_mean"]
)


# Prevent tiny negative values caused by rounding
for col in ["Excellent", "Good_only", "Fair_only", "Poor"]:
    df[col] = df[col].clip(lower=0, upper=100)


# ----------------------------------------------------
# Rating colors
# ----------------------------------------------------
rating_colors = {
    "Excellent": "#2ca25f",
    "Good_only": "#3182bd",
    "Fair_only": "#f0ad4e",
    "Poor": "#d62728"
}


# ----------------------------------------------------
# Choose evaluation domain by scenario
# ----------------------------------------------------
def choose_domain_for_scenario(data, scenario):
    """
    Select the most appropriate evaluation domain
    for each degradation scenario.

    Random/block missingness:
        corrupted_only if available.

    Additive noise:
        corrupted_only if available, otherwise full.

    Field-realistic fault:
        full if available.
    """

    scenario_data = data[
        data["scenario"] == scenario
    ]

    available_domains = set(
        scenario_data["evaluation_domain"].unique()
    )

    if scenario in ["random_missing", "block_missing"]:

        if "corrupted_only" in available_domains:
            return "corrupted_only"

        elif "gap_only" in available_domains:
            return "gap_only"

        else:
            return sorted(available_domains)[0]

    elif scenario == "additive_noise":

        if "corrupted_only" in available_domains:
            return "corrupted_only"

        elif "full" in available_domains:
            return "full"

        else:
            return sorted(available_domains)[0]

    elif scenario == "field_realistic_fault":

        if "full" in available_domains:
            return "full"

        else:
            return sorted(available_domains)[0]

    else:
        return sorted(available_domains)[0]


# ----------------------------------------------------
# Font settings
# ----------------------------------------------------
plt.rcParams["font.family"] = "serif"

plt.rcParams["font.serif"] = [
    "Times New Roman",
    "Times",
    "DejaVu Serif"
]

plt.rcParams["mathtext.fontset"] = "stix"

plt.rcParams["font.size"] = 16
plt.rcParams["axes.titlesize"] = 16
plt.rcParams["axes.labelsize"] = 16
plt.rcParams["xtick.labelsize"] = 16
plt.rcParams["ytick.labelsize"] = 16
plt.rcParams["legend.fontsize"] = 16


# ----------------------------------------------------
# Plot settings
# ----------------------------------------------------
scenario_order = [
    "random_missing",
    "block_missing",
    "additive_noise",
    "field_realistic_fault"
]

parameter_order = [
    "DO",
    "EC",
    "Temp",
    "Turbidity"
]

rating_cols = [
    "Excellent",
    "Good_only",
    "Fair_only",
    "Poor"
]

rating_labels = [
    "Excellent",
    "Good",
    "Fair",
    "Poor"
]


# ----------------------------------------------------
# Output directory
# ----------------------------------------------------
output_dir = Path("usgs_rating_figures")

output_dir.mkdir(
    exist_ok=True
)


# Keep up to 9 algorithms
top_n_algorithms = 9


# ----------------------------------------------------
# Generate one stacked bar figure per scenario
# ----------------------------------------------------
for scenario in scenario_order:

    if scenario not in df["scenario"].unique():

        print(
            f"Skipping {scenario}: not found in data"
        )

        continue


    # ------------------------------------------------
    # Select evaluation domain
    # ------------------------------------------------
    domain = choose_domain_for_scenario(
        df,
        scenario
    )


    scenario_df = df[
        (df["scenario"] == scenario)
        &
        (df["evaluation_domain"] == domain)
    ].copy()


    if scenario_df.empty:

        print(
            f"Skipping {scenario}: "
            f"no data for selected domain {domain}"
        )

        continue


    # ------------------------------------------------
    # Create 2 x 2 figure
    # ------------------------------------------------
    fig, axes = plt.subplots(
        2,
        2,
        figsize=(16, 10),
        sharey=True
    )

    axes = axes.flatten()


    # ------------------------------------------------
    # Plot each water-quality parameter
    # ------------------------------------------------
    for ax_index, (ax, parameter) in enumerate(
        zip(axes, parameter_order)
    ):

        p_df = scenario_df[
            scenario_df["parameter"] == parameter
        ].copy()


        if p_df.empty:

            ax.set_title(
                f"{parameter_display[parameter]}: no data",
                fontsize=15
            )

            ax.axis("off")

            continue


        # --------------------------------------------
        # Select best algorithms by lowest
        # Mean_accuracy_rating
        #
        # Excellent = 1
        # Good      = 2
        # Fair      = 3
        # Poor      = 4
        # --------------------------------------------
        p_df = (
            p_df
            .sort_values(
                "Mean_accuracy_rating",
                ascending=True
            )
            .head(top_n_algorithms)
        )


        # Best algorithm appears first
        p_df = p_df.sort_values(
            "Mean_accuracy_rating",
            ascending=True
        )


        x = np.arange(
            len(p_df)
        )

        bottom = np.zeros(
            len(p_df)
        )


        # --------------------------------------------
        # Stacked vertical bars
        # --------------------------------------------
        for col, label in zip(
            rating_cols,
            rating_labels
        ):

            ax.bar(
                x,
                p_df[col].values,
                bottom=bottom,
                label=label,
                color=rating_colors[col],
                edgecolor="none",
                linewidth=0
            )

            bottom += p_df[col].values


        # --------------------------------------------
        # Parameter title
        # --------------------------------------------
        ax.set_title(
            parameter_display[parameter],
            fontsize=15,
            fontweight="bold",
            pad=8
        )


        # --------------------------------------------
        # Abbreviated algorithm names
        # --------------------------------------------
        algorithm_labels = [
            algo_display.get(
                name,
                name.replace("_", " ")
            )
            for name in p_df["algorithm"].values
        ]


        ax.set_xticks(
            x
        )


        ax.set_xticklabels(
            algorithm_labels,
            rotation=20,
            ha="right",
            fontsize=16
        )


        # --------------------------------------------
        # Y-axis settings
        # Extra space above 100 for percentage labels
        # --------------------------------------------
        ax.set_ylim(
            0,
            106
        )


        ax.set_yticks(
            np.arange(
                0,
                101,
                20
            )
        )


        ax.tick_params(
            axis="y",
            labelsize=16
        )


        # --------------------------------------------
        # Y-axis title ONLY on left-hand panels
        #
        # axes[0] = DO
        # axes[1] = EC
        # axes[2] = Temperature
        # axes[3] = Turbidity
        #
        # This prevents the y-axis title from
        # appearing in the middle of the 2 x 2 figure.
        # --------------------------------------------
        if ax_index in [0, 2]:

            ax.set_ylabel(
                "Percentage of corrected observations (%)",
                fontsize=16
            )

        else:

            ax.set_ylabel("")


        # --------------------------------------------
        # Keep Good-or-better percentage above bars
        # --------------------------------------------
        for i, (_, row) in enumerate(
            p_df.iterrows()
        ):

            ax.text(
                i,
                101.2,
                f"{row['Good_or_better_rate_mean']:.1f}%",
                ha="center",
                va="bottom",
                fontsize=16
            )


        # --------------------------------------------
        # Remove subplot boundary lines
        # --------------------------------------------
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["bottom"].set_visible(False)
        ax.spines["left"].set_visible(False)


    # ------------------------------------------------
    # Common legend
    # ------------------------------------------------
    handles, labels = (
        axes[0]
        .get_legend_handles_labels()
    )


    fig.legend(
        handles,
        labels,
        loc="lower center",
        ncol=4,
        frameon=False,
        fontsize=16
    )


    # ------------------------------------------------
    # Layout
    # No scenario title
    # ------------------------------------------------
    fig.tight_layout(
        rect=[
            0,
            0.06,
            1,
            1
        ]
    )


    # ------------------------------------------------
    # Save figures
    # ------------------------------------------------
    png_path = (
        output_dir
        / f"usgs_rating_{scenario}.png"
    )

    pdf_path = (
        output_dir
        / f"usgs_rating_{scenario}.pdf"
    )


    fig.savefig(
        png_path,
        dpi=600,
        bbox_inches="tight",
        facecolor="white"
    )


    fig.savefig(
        pdf_path,
        bbox_inches="tight",
        facecolor="white"
    )


    # ------------------------------------------------
    # Display figure
    # ------------------------------------------------
    plt.show()


    print(
        f"Saved: {png_path}"
    )

    print(
        f"Saved: {pdf_path}"
    )

## Mean RMSE and MAE ranking (Figure 3 and 4)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ============================================================
# SETTINGS
# ============================================================

raw_file = "C:/Users/abadrud/2026-05-07 Update/results_fixed_rule/results_fixed_rule_raw_expanded.csv"
output_dir = Path("rank_heatmaps")
output_dir.mkdir(exist_ok=True)

parameter_order = ["Turbidity", "EC", "DO", "Temp"]
severity_order = ["moderate", "severe"]

parameter_display = {
    "Turbidity": "Turbidity",
    "EC": "EC",
    "DO": "DO",
    "Temp": "Temperature"
}

algo_display = {
    "Linear_Interpolation": "Linear",
    "Linear Interpolation": "Linear",
    "PCHIP": "PCHIP",
    "Cubic_Spline": "Spline",
    "Cubic Spline": "Spline",
    "Cubic_Spline_Interpolation": "Spline",
    "Cubic Spline Interpolation": "Spline",
    "Kalman_Filter": "Kalman",
    "Kalman Filter": "Kalman",
    "Local_Level_Kalman_Smoother": "Kalman",
    "Local-Level Kalman Smoother": "Kalman",
    "SSA_Gap_Filling": "SSA gap",
    "SSA Gap Filling": "SSA gap",
    # New gap-filling algorithms
    "Local_Moving_Mean": "Moving mean",
    "Delta_H_Weighted_Interpolation": "ΔH weighted",
    "HPWI": "HPWI"
}

plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Times New Roman", "Times", "DejaVu Serif"]
plt.rcParams["font.size"] = 12


# ============================================================
# LOAD AND CLEAN RAW RESULTS
# ============================================================

raw = pd.read_csv(raw_file)

for col in ["parameter", "scenario", "severity", "algorithm", "evaluation_domain", "status"]:
    if col in raw.columns:
        raw[col] = raw[col].astype(str)

for col in ["RMSE", "MAE"]:
    raw[col] = pd.to_numeric(raw[col], errors="coerce")


# ============================================================
# KEEP GAP-FILLING EXPERIMENTS
# ============================================================

gap = raw[raw["scenario"].isin(["random_missing", "block_missing"])].copy()

# Keep successful runs
if "status" in gap.columns:
    successful = gap["status"].str.lower().isin(["success", "successful", "ok", "completed"])
    if successful.any():
        gap = gap[successful].copy()

# Keep corrupted-only evaluation
domains = set(gap["evaluation_domain"].dropna().unique())

if "corrupted_only" in domains:
    gap = gap[gap["evaluation_domain"] == "corrupted_only"].copy()
elif "gap_only" in domains:
    gap = gap[gap["evaluation_domain"] == "gap_only"].copy()
else:
    print("WARNING: Check evaluation_domain values:", sorted(domains))

# Remove invalid RMSE/MAE
gap = gap[np.isfinite(gap["RMSE"]) & np.isfinite(gap["MAE"])].copy()


# ============================================================
# RANK ALGORITHMS WITHIN EACH MATCHED RUN
# ============================================================

rank_group = ["parameter", "storm_id", "scenario", "severity", "repetition"]

gap["RMSE_Rank"] = gap.groupby(rank_group)["RMSE"].rank(method="average", ascending=True)
gap["MAE_Rank"] = gap.groupby(rank_group)["MAE"].rank(method="average", ascending=True)

# Check number of algorithms per matched run
group_sizes = gap.groupby(rank_group)["algorithm"].nunique()

print("\nNumber of algorithms per matched experiment:")
print(group_sizes.value_counts().sort_index())


# ============================================================
# SUMMARIZE RANKS WHILE RETAINING SEVERITY
# ============================================================

rank_summary = (
    gap.groupby(["scenario", "severity", "parameter", "algorithm"], as_index=False)
       .agg(
           Mean_RMSE_Rank=("RMSE_Rank", "mean"),
           Mean_MAE_Rank=("MAE_Rank", "mean"),
           SD_RMSE_Rank=("RMSE_Rank", "std"),
           SD_MAE_Rank=("MAE_Rank", "std"),
           n_runs=("RMSE_Rank", "count")
       )
)

rank_summary.to_csv("gap_filling_RMSE_MAE_rank_summary_by_severity.csv", index=False)


# ============================================================
# PLOTTING FUNCTION
# ROWS = SEVERITY
# COLUMNS = WATER-QUALITY PARAMETERS
# ============================================================

def plot_rank_heatmap(rank_summary, scenario):

    df_s = rank_summary[rank_summary["scenario"] == scenario].copy()

    if df_s.empty:
        raise ValueError(f"No data found for {scenario}")

    metric_columns = ["Mean_RMSE_Rank", "Mean_MAE_Rank"]
    metric_labels = ["RMSE", "MAE"]

    # --------------------------------------------------------
    # Common algorithm order across severity and parameters
    # --------------------------------------------------------

    df_s["Overall_Rank"] = df_s[metric_columns].mean(axis=1)

    algorithm_order = (
        df_s.groupby("algorithm")["Overall_Rank"]
            .mean()
            .sort_values()
            .index
            .tolist()
    )

    n_algorithms = len(algorithm_order)

    # --------------------------------------------------------
    # 2 × 4 MATRIX
    # --------------------------------------------------------

    fig, axes = plt.subplots(2, 4, figsize=(18, 9), facecolor="white")
    last_im = None

    for row, severity in enumerate(severity_order):

        for col, parameter in enumerate(parameter_order):

            ax = axes[row, col]

            p_df = df_s[
                (df_s["severity"].str.lower() == severity) &
                (df_s["parameter"] == parameter)
            ].copy()

            if p_df.empty:
                ax.axis("off")
                continue

            # Same algorithm order in every panel
            p_df["algorithm"] = pd.Categorical(
                p_df["algorithm"],
                categories=algorithm_order,
                ordered=True
            )

            p_df = p_df.sort_values("algorithm")

            # Heatmap matrix
            Z = p_df[metric_columns].to_numpy(dtype=float)

            last_im = ax.imshow(
                Z,
                cmap="turbo",
                aspect="auto",
                interpolation="bilinear",
                vmin=1,
                vmax=n_algorithms
            )

            # RMSE / MAE labels
            ax.set_xticks(np.arange(len(metric_labels)))
            ax.set_xticklabels(metric_labels, fontsize=13)

            # Algorithm labels
            y_labels = [
                algo_display.get(str(a), str(a).replace("_", " "))
                for a in p_df["algorithm"]
            ]

            ax.set_yticks(np.arange(len(y_labels)))

            if col == 0:
                ax.set_yticklabels(y_labels, fontsize=13)
            else:
                ax.set_yticklabels([])

            # Parameter titles only on top row
            if row == 0:
                ax.set_title(
                    parameter_display.get(parameter, parameter),
                    fontsize=20,
                    #fontweight="bold",
                    pad=10
                )

            # Severity labels only on first column
            if col == 0:
                ax.set_ylabel(
                    severity.capitalize(),
                    fontsize=20,
                    #fontweight="bold",
                    labelpad=18
                )

            for spine in ax.spines.values():
                spine.set_visible(False)

            ax.tick_params(length=0)


    # ========================================================
    # RESERVE SPACE FOR SHARED COLORBAR
    # ========================================================

    fig.subplots_adjust(
        left=0.11,
        right=0.90,
        top=0.92,
        bottom=0.10,
        wspace=0.16,
        hspace=0.25
    )

    # Colorbar remains in SAME figure but does not overlap matrix
    cbar_ax = fig.add_axes([0.925, 0.20, 0.015, 0.60])

    cbar = fig.colorbar(last_im, cax=cbar_ax)
    cbar.set_label("Mean rank", fontsize=18, labelpad=12)
    cbar.set_ticks(np.arange(1, n_algorithms + 1))
    cbar.ax.tick_params(labelsize=15)

    # Rank 1 at top → Rank 5 at bottom
    cbar.ax.invert_yaxis()

    #fig.text(
       # 0.50,
       # 0.035,
       # "Lower rank indicates better reconstruction performance.",
      # ha="center",
      #  fontsize=14
    #)

    # ========================================================
    # SAVE
    # ========================================================

    png_path = output_dir / f"{scenario}_RMSE_MAE_rank_by_severity.png"
    pdf_path = output_dir / f"{scenario}_RMSE_MAE_rank_by_severity.pdf"

    fig.savefig(png_path, dpi=600, bbox_inches="tight", facecolor="white")
    fig.savefig(pdf_path, bbox_inches="tight", facecolor="white")

    plt.show()

    print(f"\nSaved:\n{png_path.resolve()}\n{pdf_path.resolve()}")


# ============================================================
# CREATE BOTH FIGURES
# ============================================================

plot_rank_heatmap(rank_summary, "random_missing")
plot_rank_heatmap(rank_summary, "block_missing")

##  Mean and SD of RMSE and MAE (Table 4 and 5)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# -----------------------------
# Load raw result
# -----------------------------
raw_file = r"C:\Users\abadrud\2026-05-07 Update\results_fixed_rule\results_fixed_rule_raw_expanded.csv"
# Or use:
# raw_file = "results_fixed_rule_raw.csv"

raw = pd.read_csv(raw_file)

outdir = Path("analysis_outputs")
outdir.mkdir(exist_ok=True)

print("Raw shape:", raw.shape)
print(raw["status"].value_counts(dropna=False))
print(raw[["scenario", "evaluation_domain"]].drop_duplicates())

In [ ]:
# -----------------------------
# Filter successful gap-filling results
# -----------------------------
gap_scenarios = ["random_missing", "block_missing"]
gap_domains = ["gap_only", "corrupted_only"]

gap_raw = raw[
    (raw["status"] == "ok") &
    (raw["scenario"].isin(gap_scenarios)) &
    (raw["evaluation_domain"].isin(gap_domains))
].copy()

print("Gap-filling rows:", gap_raw.shape)
print(gap_raw[["scenario", "evaluation_domain"]].drop_duplicates())
print(gap_raw["algorithm"].value_counts())

In [ ]:
gap_scenarios = ["random_missing", "block_missing"]
gap_domains = ["gap_only", "corrupted_only"]

gap_raw = raw[
    (raw["status"] == "ok") &
    (raw["scenario"].isin(gap_scenarios)) &
    (raw["evaluation_domain"].isin(gap_domains))
].copy()

mean_by_parameter_scenario_algorithm = (
    gap_raw
    .groupby(["parameter", "scenario", "algorithm"], as_index=False)
    .agg(
        RMSE_mean=("RMSE", "mean"),
        RMSE_sd=("RMSE", "std"),
        MAE_mean=("MAE", "mean"),
        MAE_sd=("MAE", "std"),
        n=("RMSE", "count")
    )
)

mean_by_parameter_scenario_algorithm = mean_by_parameter_scenario_algorithm.sort_values(
    ["scenario", "parameter", "RMSE_mean"]
)

display(mean_by_parameter_scenario_algorithm)

In [ ]:
random_raw = raw[
    (raw["status"] == "ok") &
    (raw["scenario"] == "block_missing") &
    (raw["evaluation_domain"].isin(["gap_only", "corrupted_only"]))
].copy()

random_summary = (
    random_raw
    .groupby(["algorithm", "parameter"], as_index=False)
    .agg(
        RMSE_mean=("RMSE", "mean"),
        RMSE_sd=("RMSE", "std"),
        MAE_mean=("MAE", "mean"),
        MAE_sd=("MAE", "std"),
        n=("RMSE", "count")
    )
)

display(random_summary)

## Mean Composire Rank (Figure 5 and 6)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

outdir = Path("analysis_outputs")
outdir.mkdir(exist_ok=True)

scenario_name = "additive_noise"   # change to "random_perturbation" if that is your scenario name

# Use your clean summary dataframe
df = raw[
    (raw["status"] == "ok") &
    (raw["scenario"] == scenario_name) &
    (raw["evaluation_domain"].isin(["full", "full_series"]))
].copy()

# ---------------------------------------------------------
# Multi-metric table by parameter and algorithm
# ---------------------------------------------------------
metric_by_parameter = (
    df
    .groupby(["parameter", "algorithm"], as_index=False)
    .agg(
        RMSE=("RMSE", "mean"),
        RMSE_sd=("RMSE", "std"),
        
        MAE=("MAE", "mean"),
        MAE_sd=("MAE", "std"),

        KGE=("KGE", "mean"),
        KGE_sd=("KGE", "std"),

        NSE=("NSE", "mean"),
        NSE_sd=("NSE", "std"),

        Pearson_r=("Pearson_r", "mean"),
        Pearson_r_sd=("Pearson_r", "std"),

        Peak_Error=("Peak_Error", "mean"),
        Peak_Error_sd=("Peak_Error", "std"),

        Variance_Ratio=("Variance_Ratio", "mean"),
        Variance_Ratio_sd=("Variance_Ratio", "std"),

        Shannon_Entropy_Diff=("Shannon_Entropy_Diff", "mean"),
        Shannon_Entropy_Diff_sd=("Shannon_Entropy_Diff", "std"),
    )
)

# ---------------------------------------------------------
# Add parameter-specific ranks
# ---------------------------------------------------------
metric_by_parameter["RMSE_rank"] = metric_by_parameter.groupby("parameter")["RMSE"].rank(
    ascending=True, method="min"
)

metric_by_parameter["MAE_rank"] = metric_by_parameter.groupby("parameter")["MAE"].rank(
    ascending=True, method="min"
)

metric_by_parameter["KGE_rank"] = metric_by_parameter.groupby("parameter")["KGE"].rank(
    ascending=False, method="min"
)

metric_by_parameter["NSE_rank"] = metric_by_parameter.groupby("parameter")["NSE"].rank(
    ascending=False, method="min"
)

metric_by_parameter["Pearson_rank"] = metric_by_parameter.groupby("parameter")["Pearson_r"].rank(
    ascending=False, method="min"
)

metric_by_parameter["Peak_rank"] = metric_by_parameter.groupby("parameter")["Peak_Error"].apply(
    lambda x: x.abs().rank(ascending=True, method="min")
).reset_index(level=0, drop=True)

metric_by_parameter["Variance_rank"] = metric_by_parameter.groupby("parameter")["Variance_Ratio"].apply(
    lambda x: (x - 1).abs().rank(ascending=True, method="min")
).reset_index(level=0, drop=True)

metric_by_parameter["Entropy_rank"] = metric_by_parameter.groupby("parameter")["Shannon_Entropy_Diff"].rank(
    ascending=True, method="min"
)

rank_cols = [
    "RMSE_rank",
    "MAE_rank",
    "KGE_rank",
    "NSE_rank",
    "Pearson_rank",
    "Peak_rank",
    "Variance_rank",
    "Entropy_rank",
]

metric_by_parameter["Mean_Metric_Rank"] = metric_by_parameter[rank_cols].mean(axis=1)

metric_by_parameter = metric_by_parameter.sort_values(
    ["parameter", "Mean_Metric_Rank"]
)

metric_by_parameter_round = metric_by_parameter.copy()
numeric_cols = metric_by_parameter_round.select_dtypes(include="number").columns
metric_by_parameter_round[numeric_cols] = metric_by_parameter_round[numeric_cols].round(4)

display(metric_by_parameter_round)

metric_by_parameter_round.to_csv(
    outdir / f"metric_values_and_ranks_by_parameter_{scenario_name}.csv",
    index=False
)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

figdir = outdir / "figures"
figdir.mkdir(exist_ok=True)

parameters = list(metric_by_parameter["parameter"].unique())
x_labels   = ["RMSE","MAE", "KGE", "NSE", "Pearson r", "Peak", "VR", "SE diff"]

# ── compute global rank range for consistent colour scale ─────────
all_values = []
for parameter in parameters:
    df_p = metric_by_parameter[metric_by_parameter["parameter"] == parameter]
    all_values.append(df_p.set_index("algorithm")[rank_cols].values)
vmin = min(v.min() for v in all_values)
vmax = max(v.max() for v in all_values)

# ── derive figure height from the tallest panel (most algorithms) ─
max_algos = max(
    len(metric_by_parameter[metric_by_parameter["parameter"] == p])
    for p in parameters
)
row_h  = 0.38
col_w  = 0.72
n_cols = len(rank_cols)

panel_w = n_cols * col_w
panel_h = max_algos * row_h
cbar_w  = 0.25
fig_w   = 2 * panel_w + cbar_w + 1.6
fig_h   = 2 * panel_h + 1.2

# ── layout constants (figure-fraction) ───────────────────────────
cbar_right   = 0.905
left_margin  = 0.10
right_edge   = cbar_right - 0.02
top_margin   = 0.93
bot_margin   = 0.10

total_w      = right_edge - left_margin
col_gap      = 0.08
panel_w_frac = (total_w - col_gap) / 2

total_h      = top_margin - bot_margin
row_gap      = 0.08          # ← reduce this to close row spacing
panel_h_frac = (total_h - row_gap) / 2

def panel_rect(row, col):
    x = left_margin + col * (panel_w_frac + col_gap)
    y = top_margin - (row + 1) * panel_h_frac - row * row_gap
    return [x, y, panel_w_frac, panel_h_frac]

fig  = plt.figure(figsize=(fig_w, fig_h), facecolor="white")
axes = []
ims  = []

for idx, parameter in enumerate(parameters):
    row, col = divmod(idx, 2)
    ax = fig.add_axes(panel_rect(row, col))
    ax.set_facecolor("white")
    axes.append(ax)

    df_p = (
        metric_by_parameter[metric_by_parameter["parameter"] == parameter]
        .copy()
        .sort_values("Mean_Metric_Rank")
    )
    heatmap_data = df_p.set_index("algorithm")[rank_cols]
    n_alg = len(heatmap_data)

    im = ax.imshow(
        heatmap_data.values,
        aspect="auto",
        vmin=vmin,
        vmax=vmax,
        cmap="YlOrRd",
        #interpolation="bilinear"
    )
    ims.append(im)

    ax.set_xticks(range(n_cols))
    ax.set_xticklabels(x_labels, rotation=40, ha="right", fontsize=8)
    ax.tick_params(axis="x", pad=2)

    ax.set_yticks(range(n_alg))
    ax.set_yticklabels(heatmap_data.index, fontsize=8)
    ax.tick_params(axis="y", pad=2)

    ax.set_title(f"({chr(97+idx)}) {parameter}", fontsize=10, fontweight="bold", pad=5)

    threshold = vmin + (vmax - vmin) * 0.60
    for i in range(n_alg):
        for j in range(n_cols):
            val = heatmap_data.iloc[i, j]
            fc  = "white" if val > threshold else "black"
            ax.text(j, i, f"{val:.0f}",
                    ha="center", va="center",
                    fontsize=7.5, color=fc, fontweight="bold")

    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.tick_params(length=0)

# ── shared colorbar ───────────────────────────────────────────────
cbar_ax = fig.add_axes([0.905, bot_margin, 0.018, top_margin - bot_margin])
cbar = fig.colorbar(ims[0], cax=cbar_ax)
cbar.set_label("Metric rank  (1 = best)", fontsize=9, labelpad=6)
cbar.ax.tick_params(labelsize=8)
cbar.outline.set_visible(False)

fig.suptitle(
    f"Multi-metric ranking heatmap — {scenario_name}",
    fontsize=11, fontweight="bold", y=0.975,
)

plt.savefig(
    figdir / f"multimetric_rank_heatmap_{scenario_name}_combined.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)
plt.show()

## Temporal aggregation effect (Figure 7)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("results_fixed_rule_raw_frequency_expanded_sensitivity.csv")

df["resolution"] = df["sampling_frequency"].map({
    "11min_original":"11–12 min", "30min":"30 min", "1h":"1 h"
})

algo_map = {
    "Linear_Interpolation":"Linear", "PCHIP":"PCHIP", "Cubic_Spline":"Spline",
    "Kalman_Filter":"Kalman", "SSA_Gap_Filling":"SSA gap",
    "Local_Moving_Mean":"Moving mean", "Delta_H_Weighted_Interpolation":"ΔH weighted",
    "HPWI":"HPWI", "Median_Filter":"Median filter",
    "Savitzky_Golay":"Savitzky–Golay", "LOESS":"LOESS", "SSA":"SSA",
    "Wavelet":"Wavelet", "VMD":"VMD", "EWMA":"EWMA", "CEEMDAN":"CEEMDAN"
}

df["algorithm_display"] = df["algorithm"].map(algo_map).fillna(df["algorithm"])

scenarios = ["random_missing","block_missing","additive_noise","field_realistic_fault"]
scenario_labels = {
    "random_missing":"Random missingness", "block_missing":"Block missingness",
    "additive_noise":"Additive noise", "field_realistic_fault":"Field-realistic fault"
}

parameters = ["Turbidity","EC","DO","Temp"]
parameter_labels = {"Turbidity":"Turbidity","EC":"Electrical conductivity","DO":"Dissolved oxygen","Temp":"Temperature"}

domain_map = {
    "random_missing":"corrupted_only", "block_missing":"corrupted_only",
    "additive_noise":"full", "field_realistic_fault":"full"
}

# Correct evaluation domain
parts = []
for scenario in scenarios:
    parts.append(df[(df["scenario"]==scenario) & (df["status"]=="ok") &
                    (df["evaluation_domain"]==domain_map[scenario])].copy())

a = pd.concat(parts, ignore_index=True)

# Mean RMSE at each temporal resolution
mean_rmse = a.groupby(
    ["scenario","parameter","algorithm_display","resolution"], as_index=False
)["RMSE"].mean()

# Put resolutions side-by-side
wide = mean_rmse.pivot(
    index=["scenario","parameter","algorithm_display"],
    columns="resolution", values="RMSE"
).reset_index()

wide = wide.dropna(subset=["11–12 min","30 min","1 h"])
wide = wide[wide["11–12 min"] > 0].copy()

# % change based on GROUP MEAN RMSE, not mean trial-level percentages
wide["30 min"] = 100 * (wide["30 min"] - wide["11–12 min"]) / wide["11–12 min"]
wide["1 h"] = 100 * (wide["1 h"] - wide["11–12 min"]) / wide["11–12 min"]

# Preferred algorithm order
gap_order = ["Linear","PCHIP","Spline","Kalman","SSA gap","Moving mean","ΔH weighted","HPWI"]
correction_order = ["Median filter","LOESS","SSA","Savitzky–Golay","Kalman",
                    "Wavelet","VMD","EWMA","CEEMDAN"]

fig, axes = plt.subplots(4,4,figsize=(14,14))
bar_h = 0.34

for i, scenario in enumerate(scenarios):
    preferred = gap_order if scenario in ["random_missing","block_missing"] else correction_order

    for j, parameter in enumerate(parameters):
        ax = axes[i,j]
        sub = wide[(wide["scenario"]==scenario) & (wide["parameter"]==parameter)].copy()

        # Keep every algorithm actually present
        present = sub["algorithm_display"].unique()
        algorithms = [alg for alg in preferred if alg in present]
        algorithms += [alg for alg in present if alg not in algorithms]

        sub = sub.set_index("algorithm_display").reindex(algorithms)
        y = np.arange(len(algorithms))

        ax.barh(y-bar_h/2, sub["30 min"], height=bar_h, label="30 min")
        ax.barh(y+bar_h/2, sub["1 h"], height=bar_h, label="1 h")
        ax.axvline(0, color="black", linewidth=0.8)

        # PANEL-SPECIFIC symmetric scale
        vals = np.abs(sub[["30 min","1 h"]].to_numpy(dtype=float))
        limit = np.nanmax(vals) if np.isfinite(vals).any() else 1
        limit = max(5, np.ceil(limit/5)*5)
        ax.set_xlim(-limit, limit)

        ax.set_yticks(y)
        if j == 0:
            ax.set_yticklabels(algorithms)
            ax.set_ylabel(scenario_labels[scenario], fontweight="bold", fontsize=12)
        else:
            ax.set_yticklabels([])

        ax.invert_yaxis()
        ax.grid(axis="x", alpha=0.25)
        # Remove subplot borders
        #for spine in ax.spines.values():
            #spine.set_visible(False)

        if i == 0:
            ax.set_title(parameter_labels[parameter], fontweight="bold")

        if i == 3:
            ax.set_xlabel("Change in RMSE (%)")

handles, labels = axes[0,0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=2, frameon=False)

#fig.suptitle(
   # "Effect of temporal aggregation on RMSE relative to the original 11–12 min resolution",
   # fontsize=13, fontweight="bold"
#)

fig.text(
    0.5, 0.025,
    "Negative values indicate lower RMSE; positive values indicate higher RMSE after temporal aggregation.",
    ha="center"
)

fig.tight_layout(rect=[0,0.05,1,0.96])

fig.savefig("figure_temporal_aggregation_RMSE_change.png", dpi=600, bbox_inches="tight")
fig.savefig("figure_temporal_aggregation_RMSE_change.pdf", bbox_inches="tight")

plt.show()

## Table S3 and S4

In [ ]:
import pandas as pd

# ============================================================
# COMMON ORDERS
# ============================================================

parameter_order = ["Turbidity", "EC", "DO", "Temp"]

algorithm_order = [
    "Linear_Interpolation",
    "PCHIP",
    "Cubic_Spline",
    "Kalman_Filter",
    "SSA_Gap_Filling",
        # New gap-filling algorithms
    "Local_Moving_Mean",
    "Delta_H_Weighted_Interpolation",
    "HPWI"
]

parameter_display = {
    "Turbidity": "Turbidity",
    "EC": "EC",
    "DO": "DO",
    "Temp": "Temperature"
}

algorithm_display = {
    "Linear_Interpolation": "Linear",
    "PCHIP": "PCHIP",
    "Cubic_Spline": "Spline",
    "Kalman_Filter": "Kalman",
    "SSA_Gap_Filling": "SSA gap",
        # New gap-filling algorithms
    "Local_Moving_Mean": "Moving mean",
    "Delta_H_Weighted_Interpolation": "ΔH weighted",
    "HPWI": "HPWI",
}


# ============================================================
# TABLE S4 — SEVERITY-SPECIFIC RANKS
# ============================================================

rank = pd.read_csv(
    "gap_filling_RMSE_MAE_rank_summary_by_severity.csv"
)

rank = rank[
    rank["scenario"] == "random_missing"
].copy()

rank["severity"] = rank["severity"].str.lower()

# Mean rank only — cleaner for supplementary table
rmse = rank.pivot_table(
    index=["parameter", "algorithm"],
    columns="severity",
    values="Mean_RMSE_Rank"
)

mae = rank.pivot_table(
    index=["parameter", "algorithm"],
    columns="severity",
    values="Mean_MAE_Rank"
)

table_s4 = pd.DataFrame(index=rmse.index)

table_s4["Moderate RMSE Rank"] = rmse["moderate"]
table_s4["Moderate MAE Rank"] = mae["moderate"]
table_s4["Severe RMSE Rank"] = rmse["severe"]
table_s4["Severe MAE Rank"] = mae["severe"]

table_s4 = table_s4.reset_index()

# Display names
table_s4["Parameter"] = table_s4["parameter"].map(parameter_display)
table_s4["Algorithm"] = table_s4["algorithm"].map(algorithm_display)

# Sort properly
table_s4["parameter"] = pd.Categorical(
    table_s4["parameter"],
    categories=parameter_order,
    ordered=True
)

table_s4["algorithm"] = pd.Categorical(
    table_s4["algorithm"],
    categories=algorithm_order,
    ordered=True
)

table_s4 = table_s4.sort_values(
    ["parameter", "algorithm"]
)

# Round
rank_cols = [
    "Moderate RMSE Rank",
    "Moderate MAE Rank",
    "Severe RMSE Rank",
    "Severe MAE Rank"
]

table_s4[rank_cols] = table_s4[rank_cols].round(2)

# Final column order
table_s4 = table_s4[
    ["Parameter", "Algorithm"] + rank_cols
]

table_s4.to_csv(
    "Table_S4_random_missingness_severity_ranks_CLEAN.csv",
    index=False
)

print("\nTABLE S4")
print(table_s4.to_string(index=False))


# ============================================================
# TABLE S5 — HYDROLOGIC-PHASE PERFORMANCE
# ============================================================

phase = pd.read_csv(
    "supplementary_hydrologic_phase_analysis/"
    "performance_summary_by_hydrologic_phase.csv"
)

# Normalize phase labels just in case
phase["Hydrologic_Phase"] = phase["Hydrologic_Phase"].replace({
    "Baseline": "Baseline/inter-event"
})

mae = phase.pivot_table(
    index=["parameter", "algorithm"],
    columns="Hydrologic_Phase",
    values="Mean_MAE"
)

rmse = phase.pivot_table(
    index=["parameter", "algorithm"],
    columns="Hydrologic_Phase",
    values="Mean_RMSE"
)

table_s5 = pd.DataFrame(index=mae.index)

table_s5["Baseline MAE"] = mae["Baseline/inter-event"]
table_s5["Baseline RMSE"] = rmse["Baseline/inter-event"]

table_s5["Storm-limb MAE"] = mae["Storm limb"]
table_s5["Storm-limb RMSE"] = rmse["Storm limb"]

table_s5["Peak-window MAE"] = mae["Peak window"]
table_s5["Peak-window RMSE"] = rmse["Peak window"]

table_s5 = table_s5.reset_index()

# Display names
table_s5["Parameter"] = table_s5["parameter"].map(parameter_display)
table_s5["Algorithm"] = table_s5["algorithm"].map(algorithm_display)

# Sort
table_s5["parameter"] = pd.Categorical(
    table_s5["parameter"],
    categories=parameter_order,
    ordered=True
)

table_s5["algorithm"] = pd.Categorical(
    table_s5["algorithm"],
    categories=algorithm_order,
    ordered=True
)

table_s5 = table_s5.sort_values(
    ["parameter", "algorithm"]
)

metric_cols = [
    "Baseline MAE",
    "Baseline RMSE",
    "Storm-limb MAE",
    "Storm-limb RMSE",
    "Peak-window MAE",
    "Peak-window RMSE"
]

table_s5[metric_cols] = table_s5[metric_cols].round(3)

table_s5 = table_s5[
    ["Parameter", "Algorithm"] + metric_cols
]

table_s5.to_csv(
    "Table_S5_random_missingness_hydrologic_phase_CLEAN.csv",
    index=False
)

print("\nTABLE S5")
print(table_s5.to_string(index=False))

In [ ]:
# ============================================================
# TABLE S5 — HYDROLOGIC-PHASE PERFORMANCE
# ALL 8 ALGORITHMS
#
# Reported as:
#     Mean ± SD
#
# Procedure:
# 1. Calculate MAE/RMSE within each individual degradation trial
# 2. Aggregate those trial-level metrics across storms,
#    severities, and repetitions
#
# This avoids treating every missing point as an independent
# experimental replicate.
# ============================================================

import pandas as pd
import numpy as np


# ============================================================
# READ NEW 8-ALGORITHM POINTWISE FILE
# ============================================================

pointwise_file = (
    "supplementary_random_missing_hydrologic_phase_8alg/"
    "random_missing_pointwise_reconstruction_8alg.csv"
)

phase = pd.read_csv(
    pointwise_file
)


# ============================================================
# BASIC CLEANING
# ============================================================

for col in [
    "parameter",
    "algorithm",
    "hydrologic_phase",
    "severity"
]:

    if col in phase.columns:

        phase[col] = (
            phase[col]
            .astype(str)
            .str.strip()
        )


# Normalize phase labels just in case
phase["hydrologic_phase"] = (
    phase["hydrologic_phase"]
    .replace({
        "Baseline": "Baseline/inter-event",
        "baseline": "Baseline/inter-event",
        "Storm Limb": "Storm limb",
        "Peak Window": "Peak window"
    })
)


# ============================================================
# VERIFY ALL 8 ALGORITHMS ARE PRESENT
# ============================================================

print("\nAlgorithms found in hydrologic-phase pointwise file:")

alg_check = sorted(
    phase["algorithm"]
    .dropna()
    .unique()
)

for i, alg in enumerate(
    alg_check,
    start=1
):

    print(
        f"{i}. "
        f"{algorithm_display.get(alg, alg)} "
        f"[{alg}]"
    )


print(
    "\nNumber of algorithms:",
    len(alg_check)
)


if len(alg_check) != 8:

    missing_algs = [
        alg
        for alg in algorithm_order
        if alg not in alg_check
    ]

    raise ValueError(
        "Expected 8 algorithms in the pointwise phase file. "
        f"Missing: {missing_algs}"
    )


# ============================================================
# ENSURE ERROR VARIABLES EXIST
# ============================================================

phase["Error"] = (
    phase["reconstructed"]
    -
    phase["reference"]
)

phase["Absolute_Error"] = np.abs(
    phase["Error"]
)

phase["Squared_Error"] = (
    phase["Error"] ** 2
)


# ============================================================
# TRIAL-LEVEL PERFORMANCE
#
# Experimental unit:
# parameter × algorithm × storm × severity × repetition × seed
# × hydrologic phase
# ============================================================

trial_phase = (

    phase

    .groupby(
        [
            "parameter",
            "algorithm",
            "storm_id",
            "severity",
            "repetition",
            "seed",
            "hydrologic_phase"
        ],
        as_index=False
    )

    .agg(

        n_points=(
            "Absolute_Error",
            "size"
        ),

        MAE=(
            "Absolute_Error",
            "mean"
        ),

        MSE=(
            "Squared_Error",
            "mean"
        )
    )
)


# Convert MSE to RMSE
trial_phase["RMSE"] = np.sqrt(
    trial_phase["MSE"]
)


# ============================================================
# SAVE TRIAL-LEVEL RESULTS
# ============================================================

trial_phase.to_csv(
    "Table_S5_trial_level_hydrologic_phase_metrics.csv",
    index=False
)


# ============================================================
# AGGREGATE ACROSS TRIALS
#
# Pooled across moderate + severe because your previous
# Table S5 was not severity-specific.
# ============================================================

phase_summary = (

    trial_phase

    .groupby(
        [
            "parameter",
            "algorithm",
            "hydrologic_phase"
        ],
        as_index=False
    )

    .agg(

        N_trials=(
            "MAE",
            "size"
        ),

        Mean_MAE=(
            "MAE",
            "mean"
        ),

        SD_MAE=(
            "MAE",
            "std"
        ),

        Mean_RMSE=(
            "RMSE",
            "mean"
        ),

        SD_RMSE=(
            "RMSE",
            "std"
        )
    )
)


# ============================================================
# OPTIONAL SAVE OF NUMERIC SUMMARY
# ============================================================

phase_summary.to_csv(
    "Table_S5_hydrologic_phase_numeric_summary_8alg.csv",
    index=False
)


# ============================================================
# PIVOT MEAN AND SD SEPARATELY
# ============================================================

mae_mean = phase_summary.pivot_table(
    index=[
        "parameter",
        "algorithm"
    ],
    columns="hydrologic_phase",
    values="Mean_MAE"
)


mae_sd = phase_summary.pivot_table(
    index=[
        "parameter",
        "algorithm"
    ],
    columns="hydrologic_phase",
    values="SD_MAE"
)


rmse_mean = phase_summary.pivot_table(
    index=[
        "parameter",
        "algorithm"
    ],
    columns="hydrologic_phase",
    values="Mean_RMSE"
)


rmse_sd = phase_summary.pivot_table(
    index=[
        "parameter",
        "algorithm"
    ],
    columns="hydrologic_phase",
    values="SD_RMSE"
)


# ============================================================
# FORMAT FUNCTION:
# Mean ± SD
# ============================================================

def mean_sd_string(
    mean_value,
    sd_value,
    decimals=3
):

    if (
        pd.isna(mean_value)
        or
        pd.isna(sd_value)
    ):

        return ""

    return (
        f"{mean_value:.{decimals}f} "
        f"± "
        f"{sd_value:.{decimals}f}"
    )


# ============================================================
# BUILD TABLE S5
# ============================================================

table_s5 = pd.DataFrame(
    index=mae_mean.index
)


# ------------------------------------------------------------
# BASELINE
# ------------------------------------------------------------

table_s5[
    "Baseline MAE"
] = [

    mean_sd_string(
        mae_mean.loc[idx, "Baseline/inter-event"],
        mae_sd.loc[idx, "Baseline/inter-event"]
    )

    for idx in table_s5.index
]


table_s5[
    "Baseline RMSE"
] = [

    mean_sd_string(
        rmse_mean.loc[idx, "Baseline/inter-event"],
        rmse_sd.loc[idx, "Baseline/inter-event"]
    )

    for idx in table_s5.index
]


# ------------------------------------------------------------
# STORM LIMB
# ------------------------------------------------------------

table_s5[
    "Storm-limb MAE"
] = [

    mean_sd_string(
        mae_mean.loc[idx, "Storm limb"],
        mae_sd.loc[idx, "Storm limb"]
    )

    for idx in table_s5.index
]


table_s5[
    "Storm-limb RMSE"
] = [

    mean_sd_string(
        rmse_mean.loc[idx, "Storm limb"],
        rmse_sd.loc[idx, "Storm limb"]
    )

    for idx in table_s5.index
]


# ------------------------------------------------------------
# PEAK WINDOW
# ------------------------------------------------------------

table_s5[
    "Peak-window MAE"
] = [

    mean_sd_string(
        mae_mean.loc[idx, "Peak window"],
        mae_sd.loc[idx, "Peak window"]
    )

    for idx in table_s5.index
]


table_s5[
    "Peak-window RMSE"
] = [

    mean_sd_string(
        rmse_mean.loc[idx, "Peak window"],
        rmse_sd.loc[idx, "Peak window"]
    )

    for idx in table_s5.index
]


# ============================================================
# RESET INDEX
# ============================================================

table_s5 = table_s5.reset_index()


# ============================================================
# DISPLAY NAMES
# ============================================================

table_s5["Parameter"] = (
    table_s5["parameter"]
    .map(parameter_display)
)


table_s5["Algorithm"] = (
    table_s5["algorithm"]
    .map(algorithm_display)
)


# Safety fallback
table_s5["Parameter"] = (
    table_s5["Parameter"]
    .fillna(
        table_s5["parameter"]
    )
)


table_s5["Algorithm"] = (
    table_s5["Algorithm"]
    .fillna(
        table_s5["algorithm"]
    )
)


# ============================================================
# SORT PARAMETER AND ALGORITHM
# ============================================================

table_s5["parameter"] = pd.Categorical(
    table_s5["parameter"],
    categories=parameter_order,
    ordered=True
)


table_s5["algorithm"] = pd.Categorical(
    table_s5["algorithm"],
    categories=algorithm_order,
    ordered=True
)


table_s5 = (
    table_s5
    .sort_values(
        [
            "parameter",
            "algorithm"
        ]
    )
)


# ============================================================
# FINAL COLUMN ORDER
# ============================================================

metric_cols = [

    "Baseline MAE",
    "Baseline RMSE",

    "Storm-limb MAE",
    "Storm-limb RMSE",

    "Peak-window MAE",
    "Peak-window RMSE"
]


table_s5 = table_s5[
    [
        "Parameter",
        "Algorithm"
    ]
    +
    metric_cols
]


# ============================================================
# SAVE CLEAN TABLE
# ============================================================

table_s5.to_csv(
    "Table_S5_random_missingness_hydrologic_phase_mean_SD_CLEAN.csv",
    index=False
)


# ============================================================
# PRINT
# ============================================================

print(
    "\n============================================"
)

print(
    "TABLE S5 — RANDOM MISSINGNESS"
)

print(
    "Hydrologic-phase performance (Mean ± SD)"
)

print(
    "============================================\n"
)


print(
    table_s5.to_string(
        index=False
    )
)

## Table S5 and S6

In [ ]:
import pandas as pd

# ============================================================
# COMMON DISPLAY ORDER
# ============================================================

parameter_order = ["Turbidity", "EC", "DO", "Temp"]

algorithm_order = [
    "Linear_Interpolation",
    "PCHIP",
    "Cubic_Spline",
    "Kalman_Filter",
    "SSA_Gap_Filling",
    "Local_Moving_Mean",
    "Delta_H_Weighted_Interpolation",
    "HPWI",
]

parameter_display = {
    "Turbidity": "Turbidity",
    "EC": "EC",
    "DO": "DO",
    "Temp": "Temperature"
}

algorithm_display = {
    "Linear_Interpolation": "Linear",
    "PCHIP": "PCHIP",
    "Cubic_Spline": "Spline",
    "Kalman_Filter": "Kalman",
    "SSA_Gap_Filling": "SSA gap",
            # New gap-filling algorithms
    "Local_Moving_Mean": "Moving mean",
    "Delta_H_Weighted_Interpolation": "ΔH weighted",
    "HPWI": "HPWI"
}


# ============================================================
# TABLE S6
# SEVERITY-SPECIFIC RANKS — BLOCK MISSINGNESS
# ============================================================

rank = pd.read_csv("gap_filling_RMSE_MAE_rank_summary_by_severity.csv")

rank = rank[rank["scenario"] == "block_missing"].copy()
rank["severity"] = rank["severity"].str.lower()

rmse = rank.pivot_table(
    index=["parameter", "algorithm"],
    columns="severity",
    values="Mean_RMSE_Rank"
)

mae = rank.pivot_table(
    index=["parameter", "algorithm"],
    columns="severity",
    values="Mean_MAE_Rank"
)

table_s6 = pd.DataFrame(index=rmse.index)

table_s6["Moderate RMSE Rank"] = rmse["moderate"]
table_s6["Moderate MAE Rank"] = mae["moderate"]
table_s6["Severe RMSE Rank"] = rmse["severe"]
table_s6["Severe MAE Rank"] = mae["severe"]

table_s6 = table_s6.reset_index()

# Display names
table_s6["Parameter"] = table_s6["parameter"].map(parameter_display)
table_s6["Algorithm"] = table_s6["algorithm"].map(algorithm_display)

# Sort
table_s6["parameter"] = pd.Categorical(
    table_s6["parameter"],
    categories=parameter_order,
    ordered=True
)

table_s6["algorithm"] = pd.Categorical(
    table_s6["algorithm"],
    categories=algorithm_order,
    ordered=True
)

table_s6 = table_s6.sort_values(["parameter", "algorithm"])

rank_cols = [
    "Moderate RMSE Rank",
    "Moderate MAE Rank",
    "Severe RMSE Rank",
    "Severe MAE Rank"
]

table_s6[rank_cols] = table_s6[rank_cols].round(2)

table_s6 = table_s6[
    ["Parameter", "Algorithm"] + rank_cols
]

table_s6.to_csv(
    "Table_S6_block_missingness_severity_ranks.csv",
    index=False
)

print("\nTABLE S6 — BLOCK MISSINGNESS SEVERITY RANKS")
print(table_s6.to_string(index=False))


# ============================================================
# TABLE S7
# HYDROLOGIC-PHASE PERFORMANCE — BLOCK MISSINGNESS
# ============================================================

phase = pd.read_csv(
    "supplementary_block_missingness_analysis/"
    "block_performance_summary_by_hydrologic_phase.csv"
)

phase["Hydrologic_Phase"] = phase["Hydrologic_Phase"].replace({
    "Baseline": "Baseline/inter-event"
})

# ------------------------------------------------------------
# Create Mean ± SD strings
# ------------------------------------------------------------

phase["MAE ± SD"] = phase.apply(
    lambda r: f"{r['Mean_MAE']:.3f} ± {r['SD_MAE']:.3f}",
    axis=1
)

phase["RMSE ± SD"] = phase.apply(
    lambda r: f"{r['Mean_RMSE']:.3f} ± {r['SD_RMSE']:.3f}",
    axis=1
)

mae = phase.pivot_table(
    index=["parameter", "algorithm"],
    columns="Hydrologic_Phase",
    values="MAE ± SD",
    aggfunc="first"
)

rmse = phase.pivot_table(
    index=["parameter", "algorithm"],
    columns="Hydrologic_Phase",
    values="RMSE ± SD",
    aggfunc="first"
)

table_s7 = pd.DataFrame(index=mae.index)

table_s7["Baseline MAE"] = mae["Baseline/inter-event"]
table_s7["Baseline RMSE"] = rmse["Baseline/inter-event"]

table_s7["Storm-limb MAE"] = mae["Storm limb"]
table_s7["Storm-limb RMSE"] = rmse["Storm limb"]

table_s7["Peak-window MAE"] = mae["Peak window"]
table_s7["Peak-window RMSE"] = rmse["Peak window"]

table_s7 = table_s7.reset_index()

# Display names
table_s7["Parameter"] = table_s7["parameter"].map(parameter_display)
table_s7["Algorithm"] = table_s7["algorithm"].map(algorithm_display)

# Sort
table_s7["parameter"] = pd.Categorical(
    table_s7["parameter"],
    categories=parameter_order,
    ordered=True
)

table_s7["algorithm"] = pd.Categorical(
    table_s7["algorithm"],
    categories=algorithm_order,
    ordered=True
)

table_s7 = table_s7.sort_values(["parameter", "algorithm"])

performance_cols = [
    "Baseline MAE",
    "Baseline RMSE",
    "Storm-limb MAE",
    "Storm-limb RMSE",
    "Peak-window MAE",
    "Peak-window RMSE"
]

table_s7 = table_s7[
    ["Parameter", "Algorithm"] + performance_cols
]

table_s7.to_csv(
    "Table_S7_block_missingness_hydrologic_phase.csv",
    index=False
)

print("\nTABLE S7 — BLOCK MISSINGNESS HYDROLOGIC PHASE")
print(table_s7.to_string(index=False))

## Figure S1

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from pathlib import Path

# ============================================================
# SETTINGS
# ============================================================

storm_ids = [1, 2, 3, 4]       # change only if your IDs differ
radar_column = "Radar"

# Classification settings
BASELINE_QUANTILE = 0.20       # lower part of inverted stage used to estimate baseline
ONSET_FRACTION = 0.04
END_FRACTION = 0.025
RETURN_HOURS = 3.0 # rise above baseline required for active response
MERGE_GAP_HOURS = 2.0          # short returns toward baseline remain same storm episode
MIN_EPISODE_HOURS = 1.0        # ignore very short excursions
MIN_PEAK_DISTANCE_HOURS = 0.75
PEAK_PROMINENCE_FRACTION = 0.15
PEAK_WINDOW_FRACTION = 0.75    # upper 15% of local peak height = peak window
SMOOTH_HOURS = 0.75

output_dir = Path("storm_phase_diagnostic")
output_dir.mkdir(exist_ok=True)


# ============================================================
# GET STORM DATAFRAME
# Assumes all_storms is a dictionary of DataFrames
# ============================================================

def get_storm_dataframe(all_storms, storm_id):
    candidates = [storm_id, str(storm_id), f"Storm_{storm_id}", f"Storm {storm_id}"]

    for key in candidates:
        if key in all_storms:
            obj = all_storms[key]

            if isinstance(obj, pd.DataFrame):
                return obj.copy()

            if isinstance(obj, dict):
                for value in obj.values():
                    if isinstance(value, pd.DataFrame):
                        return value.copy()

    raise KeyError(f"Could not find Storm {storm_id} in all_storms.")


# ============================================================
# ESTIMATE SAMPLING INTERVAL
# ============================================================

def estimate_dt_minutes(index):
    if isinstance(index, pd.DatetimeIndex) and len(index) > 2:
        dt = np.diff(index.asi8) / 60e9
        dt = dt[np.isfinite(dt) & (dt > 0)]

        if len(dt):
            return float(np.median(dt))

    # Your records are approximately 11–12 min
    return 11.5


# ============================================================
# FILL SHORT FALSE GAPS INSIDE ACTIVE RESPONSE
#
# This prevents a shallow inter-peak trough from being
# incorrectly labelled as baseline.
# ============================================================

def fill_short_false_runs(mask, max_gap):
    mask = np.asarray(mask, dtype=bool).copy()
    n = len(mask)
    i = 0

    while i < n:
        if mask[i]:
            i += 1
            continue

        start = i

        while i < n and not mask[i]:
            i += 1

        end = i
        run_length = end - start

        left_active = start > 0 and mask[start - 1]
        right_active = end < n and mask[end]

        if left_active and right_active and run_length <= max_gap:
            mask[start:end] = True

    return mask


# ============================================================
# REMOVE VERY SHORT ACTIVE EPISODES
# ============================================================

def remove_short_true_runs(mask, min_length):
    mask = np.asarray(mask, dtype=bool).copy()
    n = len(mask)
    i = 0

    while i < n:
        if not mask[i]:
            i += 1
            continue

        start = i

        while i < n and mask[i]:
            i += 1

        end = i

        if (end - start) < min_length:
            mask[start:end] = False

    return mask


# ============================================================
# LABEL ACTIVE EPISODES
# ============================================================

def label_episodes(mask):
    episode_id = np.zeros(len(mask), dtype=int)
    episode = 0
    i = 0

    while i < len(mask):
        if not mask[i]:
            i += 1
            continue

        episode += 1
        start = i

        while i < len(mask) and mask[i]:
            i += 1

        episode_id[start:i] = episode

    return episode_id


# ============================================================
# CLASSIFY ONE STORM WINDOW
# ============================================================

def classify_storm_phases(storm_df, radar_column="Radar"):

    radar = pd.to_numeric(storm_df[radar_column], errors="coerce").copy()

    # Same radar cleaning logic used previously
    radar[(np.isclose(radar, 0, atol=1.0)) | (radar > 5)] = np.nan

    dt_min = estimate_dt_minutes(storm_df.index)

    # Interpolate only very short radar gaps for classification
    radar_work = radar.interpolate(limit=2, limit_direction="both")

    # --------------------------------------------------------
    # Invert:
    # lower raw radar reading = higher water level
    # --------------------------------------------------------

    inverted = radar_work.max() - radar_work

    # Robust normalization
    valid = inverted.dropna()

    q02 = valid.quantile(0.02)
    q98 = valid.quantile(0.98)

    if not np.isfinite(q98 - q02) or (q98 - q02) <= 0:
        q02 = valid.min()
        q98 = valid.max()

    stage = ((inverted - q02) / (q98 - q02)).clip(0, 1)

    # --------------------------------------------------------
    # Smooth stage only for phase detection
    # --------------------------------------------------------

    smooth_points = max(3, int(round(SMOOTH_HOURS * 60 / dt_min)))

    if smooth_points % 2 == 0:
        smooth_points += 1

    smooth = stage.rolling(
        smooth_points,
        center=True,
        min_periods=1
    ).median()

    # --------------------------------------------------------
    # Estimate baseline and active-response threshold
    # --------------------------------------------------------

# --------------------------------------------------------
# Estimate baseline and response range
# --------------------------------------------------------

    baseline = smooth.quantile(BASELINE_QUANTILE)
    upper = smooth.quantile(0.95)

    response_range = upper - baseline

    # Separate thresholds for event onset and recession/end
    onset_threshold = baseline + ONSET_FRACTION * response_range
    end_threshold = baseline + END_FRACTION * response_range

    # Number of consecutive observations required near baseline
    # before an active event is considered finished
    return_points = max(1, int(round(RETURN_HOURS * 60 / dt_min)))

    smooth_values = smooth.to_numpy(dtype=float)
    active = np.zeros(len(smooth_values), dtype=bool)

    in_event = False
    below_end_count = 0

    for i, value in enumerate(smooth_values):

        if not np.isfinite(value):
            # If stage is missing, retain the current state rather than
            # abruptly terminating an event.
            if in_event:
                active[i] = True
            continue

        # ----------------------------------------------------
        # EVENT HAS NOT STARTED
        # ----------------------------------------------------
        if not in_event:

            if value >= onset_threshold:
                in_event = True
                below_end_count = 0
                active[i] = True

        # ----------------------------------------------------
        # EVENT IS ACTIVE
        # ----------------------------------------------------
        else:

            active[i] = True

            if value <= end_threshold:
                below_end_count += 1
            else:
                below_end_count = 0

            # End only after a sustained return toward baseline
            if below_end_count >= return_points:

                end_start = i - return_points + 1

                # The near-baseline observations used to establish
                # the return are classified back as baseline.
                active[end_start:i + 1] = False

                in_event = False
                below_end_count = 0

    # --------------------------------------------------------
    # Convert time settings to number of observations
    # --------------------------------------------------------

    merge_gap_points = max(1, int(round(MERGE_GAP_HOURS * 60 / dt_min)))
    min_episode_points = max(2, int(round(MIN_EPISODE_HOURS * 60 / dt_min)))
    min_peak_distance = max(1, int(round(MIN_PEAK_DISTANCE_HOURS * 60 / dt_min)))

    # --------------------------------------------------------
    # Prevent inter-peak valleys from becoming baseline
    # --------------------------------------------------------

    active = fill_short_false_runs(active, merge_gap_points)
    active = remove_short_true_runs(active, min_episode_points)

    episode_id = label_episodes(active)

    # --------------------------------------------------------
    # Initial three-class assignment
    # --------------------------------------------------------

    phase = np.full(len(stage), "Baseline", dtype=object)
    phase[active] = "Storm limb"

    peak_mask = np.zeros(len(stage), dtype=bool)
    peak_indices = []

    # ========================================================
    # DETECT MULTIPLE PEAKS WITHIN EACH ACTIVE EPISODE
    # ========================================================

    for episode in range(1, episode_id.max() + 1):

        idx = np.where(episode_id == episode)[0]

        if len(idx) < 3:
            continue

        segment = smooth.iloc[idx].to_numpy()

        episode_baseline = max(baseline, np.nanmin(segment))
        episode_peak = np.nanmax(segment)
        episode_amplitude = episode_peak - episode_baseline

        if not np.isfinite(episode_amplitude) or episode_amplitude <= 0:
            continue

        prominence = PEAK_PROMINENCE_FRACTION * episode_amplitude

        local_peaks, properties = find_peaks(
            segment,
            prominence=prominence,
            distance=min_peak_distance
        )

        # If no prominent peak is found, retain the main episode maximum
        if len(local_peaks) == 0:
            local_peaks = np.array([int(np.nanargmax(segment))])

        for local_peak in local_peaks:

            global_peak = idx[local_peak]
            peak_indices.append(global_peak)

            peak_value = smooth.iloc[global_peak]

            # ------------------------------------------------
            # Dynamic peak window
            #
            # Rather than ± fixed observations, identify the
            # upper portion of each individual peak.
            # ------------------------------------------------

            peak_threshold = (
                episode_baseline
                + PEAK_WINDOW_FRACTION
                * (peak_value - episode_baseline)
            )

            left = local_peak
            right = local_peak

            while left > 0 and segment[left - 1] >= peak_threshold:
                left -= 1

            while right < len(segment) - 1 and segment[right + 1] >= peak_threshold:
                right += 1

            peak_global_idx = idx[left:right + 1]
            peak_mask[peak_global_idx] = True

    phase[peak_mask] = "Peak window"

    # ========================================================
    # RETURN CLASSIFICATION
    # ========================================================

    result = pd.DataFrame({
        "Radar_raw": radar,
        "Stage_normalized": stage,
        "Stage_smoothed": smooth,
        "Active_response": active,
        "Episode_ID": episode_id,
        "Phase": phase
    }, index=storm_df.index)

    info = {
        "dt_minutes": dt_min,
        "baseline": baseline,
        "onset_threshold": onset_threshold,
        "end_threshold": end_threshold,
        "n_episodes": int(episode_id.max()),
        "n_peaks": len(peak_indices),
        "peak_indices": peak_indices
    }

    return result, info


# ============================================================
# RUN CLASSIFICATION FOR FOUR ORIGINAL STORM WINDOWS
# ============================================================

classified_storms = {}
summary_rows = []

for storm_id in storm_ids:

    storm_df = get_storm_dataframe(all_storms, storm_id)

    classified, info = classify_storm_phases(
        storm_df,
        radar_column=radar_column
    )

    classified_storms[storm_id] = classified

    counts = classified["Phase"].value_counts(normalize=True) * 100

    summary_rows.append({
        "Storm": storm_id,
        "Sampling_interval_min": info["dt_minutes"],
        "Episodes": info["n_episodes"],
        "Peaks": info["n_peaks"],
        "Baseline_%": counts.get("Baseline", 0),
        "Storm_limb_%": counts.get("Storm limb", 0),
        "Peak_window_%": counts.get("Peak window", 0)
    })

    classified.to_csv(
        output_dir / f"Storm_{storm_id}_phase_classification.csv"
    )


summary = pd.DataFrame(summary_rows)

print("\nStorm-phase classification summary:")
print(summary.round(1).to_string(index=False))

summary.to_csv(
    output_dir / "storm_phase_summary.csv",
    index=False
)


# ============================================================
# DIAGNOSTIC 2 × 2 FIGURE
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(15, 9), facecolor="white")
axes = axes.flatten()

phase_order = ["Baseline", "Storm limb", "Peak window"]
cmap = plt.get_cmap("tab10")
phase_colors = {phase: cmap(i) for i, phase in enumerate(phase_order)}

for ax, storm_id in zip(axes, storm_ids):

    classified = classified_storms[storm_id]

    # Convert plotting variables
    if isinstance(classified.index, pd.DatetimeIndex):
        x = classified.index.to_pydatetime()
    else:
        x = np.asarray(classified.index)

    smooth = classified["Stage_smoothed"].to_numpy(dtype=float)
    phase_values = classified["Phase"].to_numpy()

    # Plot underlying smoothed stage in gray so Baseline does not blend in
    ax.plot(
        x,
        smooth,
        color="0.5",
        linewidth=1.2,
        alpha=0.8,
        label="Smoothed stage"
    )

    # Plot the three classifications
    for phase in phase_order:

        mask = phase_values == phase

        ax.scatter(
            np.asarray(x)[mask],
            smooth[mask],
            s=18,
            alpha=0.80,
            color=phase_colors[phase],
            label=phase
        )

    n_episodes = classified["Episode_ID"].max()

    ax.set_title(
        f"Storm {storm_id} | {n_episodes} episode(s)",
        fontsize=15,
        fontweight="bold"
    )

    ax.set_ylabel(
        "Water level (m)",
        fontsize=15
    )

    ax.tick_params(axis="both", labelsize=10)

    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)


# Shared legend
handles, labels = axes[0].get_legend_handles_labels()
unique = dict(zip(labels, handles))

fig.legend(
    unique.values(),
    unique.keys(),
    loc="lower center",
    ncol=4,
    frameon=False,
    fontsize=15,
    bbox_to_anchor=(0.5, 0.015)
)

fig.subplots_adjust(
    left=0.08,
    right=0.98,
    top=0.94,
    bottom=0.10,
    wspace=0.20,
    hspace=0.28
)

fig.savefig(
    output_dir / "storm_phase_classification_diagnostic.png",
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

fig.savefig(
    output_dir / "storm_phase_classification_diagnostic.pdf",
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

## Figure S3

In [ ]:
# ============================================================
# POINTWISE HYDROLOGIC-PHASE ANALYSIS
# 8 GAP-FILLING ALGORITHMS
#
# Scenarios:
#   random_missing
#   block_missing
#
# Hydrologic phases:
#   Baseline/inter-event
#   Storm limb
#   Peak window
#
# Algorithms:
#   1. Linear
#   2. PCHIP
#   3. Cubic Spline
#   4. Moving Mean
#   5. Delta-H Weighted
#   6. HPWI
#   7. Kalman
#   8. SSA Gap Filling
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from matplotlib.lines import Line2D
from pathlib import Path


# ============================================================
# SETTINGS
# ============================================================

results_file = (
    r"C:\Users\abadrud\2026-05-07 Update"
    r"\results_fixed_rule\expanded_gap_experiment"
    r"\results_fixed_rule_raw_expanded_gap_8alg.csv"
)

# ------------------------------------------------------------
# SELECT SCENARIO
# ------------------------------------------------------------

scenario = "random_missing"

# To run block missingness later:
# scenario = "block_missing"


output_dir = Path(
    f"supplementary_{scenario}_hydrologic_phase_8alg"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# PARAMETER ORDER
# ============================================================

parameter_order = [
    "Turbidity",
    "EC",
    "DO",
    "Temp"
]


parameter_display = {
    "Turbidity": "Turbidity",
    "EC": "EC",
    "DO": "DO",
    "Temp": "Temperature"
}


# ============================================================
# SHORT ALGORITHM DISPLAY NAMES
# ============================================================

algo_display = {

    "Linear_Interpolation":
        "Linear",

    "PCHIP":
        "PCHIP",

    "Cubic_Spline":
        "Spline",

    "Cubic_Spline_Interpolation":
        "Spline",

    "Local_Moving_Mean":
        "Moving mean",

    "Delta_H_Weighted_Interpolation":
        "ΔH weighted",

    "HPWI":
        "HPWI",

    "Kalman_Filter":
        "Kalman",

    "Local_Level_Kalman_Smoother":
        "Kalman",

    "SSA_Gap_Filling":
        "SSA gap"
}


# ============================================================
# FORCE FINAL 8-ALGORITHM ORDER
# ============================================================

algorithm_order = [

    "Linear_Interpolation",

    "PCHIP",

    "Cubic_Spline",

    "Local_Moving_Mean",

    "Delta_H_Weighted_Interpolation",

    "HPWI",

    "Kalman_Filter",

    "SSA_Gap_Filling"
]


print("\n========================================")
print("ALGORITHMS REQUESTED")
print("========================================")

for i, algorithm in enumerate(
    algorithm_order,
    start=1
):

    print(
        f"{i}. "
        f"{algo_display.get(algorithm, algorithm)} "
        f"[{algorithm}]"
    )


print(
    "\nTotal algorithms requested:",
    len(algorithm_order)
)


assert len(algorithm_order) == 8


# ============================================================
# HYDROLOGIC PHASE ORDER
# ============================================================

phase_order = [

    "Baseline/inter-event",

    "Storm limb",

    "Peak window"
]


# ============================================================
# REQUIRED OBJECTS FROM ORIGINAL EXPERIMENT
# ============================================================

required_objects = [

    "all_storms",

    "classified_storms",

    "get_reference_signal",

    "get_storm_dataframe",

    "DEGRADATION_FUNCTIONS",

    "choose_algorithm_pool",

    "match_length"
]


missing_objects = [

    name

    for name in required_objects

    if name not in globals()
]


if missing_objects:

    raise NameError(

        "Run the original experiment and hydrologic "
        "classification setup cells first.\n\n"

        "Missing objects:\n"

        + ", ".join(missing_objects)
    )


# ============================================================
# BASIC HELPERS
# ============================================================

def storm_key(value):

    try:

        return int(
            float(value)
        )

    except Exception:

        return value


def normalize_name(value):

    return (

        str(value)

        .strip()

        .lower()

        .replace("_", " ")

        .replace("-", " ")
    )


# ============================================================
# PARAMETER COLUMN ALIASES
# ============================================================

parameter_aliases = {

    "Turbidity": [
        "Turbidity"
    ],

    "EC": [
        "EC",
        "Specific Conductance",
        "Specific_Conductance",
        "Conductivity"
    ],

    "DO": [
        "DO",
        "Dissolved Oxygen",
        "Dissolved_Oxygen"
    ],

    "Temp": [
        "Temp",
        "Temperature",
        "Water Temperature",
        "Water_Temperature"
    ]
}


def find_parameter_column(
    df,
    parameter
):

    normalized = {

        normalize_name(c): c

        for c in df.columns
    }


    for alias in parameter_aliases[
        parameter
    ]:

        key = normalize_name(
            alias
        )

        if key in normalized:

            return normalized[
                key
            ]


    return None


# ============================================================
# FIND HYDROLOGIC PHASE COLUMN
# ============================================================

def find_phase_column(df):

    preferred = [

        "hydrologic_phase",

        "hydrologic phase",

        "storm_phase",

        "storm phase",

        "phase"
    ]


    normalized = {

        normalize_name(c): c

        for c in df.columns
    }


    for name in preferred:

        key = normalize_name(
            name
        )

        if key in normalized:

            return normalized[
                key
            ]


    # Fallback
    for col in df.columns:

        if "phase" in normalize_name(
            col
        ):

            return col


    raise ValueError(

        "Could not identify hydrologic-phase column.\n\n"

        f"Available columns:\n"
        f"{df.columns.tolist()}"
    )


# ============================================================
# STANDARDIZE PHASE NAMES
# ============================================================

def standardize_phase(value):

    text = normalize_name(
        value
    )


    if "peak" in text:

        return "Peak window"


    if (
        "limb" in text
        or
        "active" in text
        or
        "storm" in text
    ):

        return "Storm limb"


    if (

        "baseline" in text

        or
        "inter event" in text

        or
        "interevent" in text

    ):

        return "Baseline/inter-event"


    return str(
        value
    )


# ============================================================
# FIND STAGE COLUMN
# ============================================================

def find_stage_column(df):

    normalized = {

        normalize_name(c): c

        for c in df.columns
    }


    # We know this was selected previously
    preferred = [

        "stage smoothed",

        "stage_smoothed",

        "smoothed stage",

        "water level smoothed",

        "stage normalized smoothed",

        "normalized stage smoothed",

        "stage normalized",

        "water level",

        "stage"
    ]


    for name in preferred:

        key = normalize_name(
            name
        )

        if key in normalized:

            col = normalized[
                key
            ]

            numeric = pd.to_numeric(
                df[col],
                errors="coerce"
            )

            if numeric.notna().sum() > 5:

                return col


    # Intelligent fallback
    candidates = []


    for col in df.columns:

        key = normalize_name(
            col
        )


        if not any(

            term in key

            for term in [
                "stage",
                "water level",
                "radar"
            ]
        ):

            continue


        numeric = pd.to_numeric(
            df[col],
            errors="coerce"
        )


        if numeric.notna().sum() <= 5:

            continue


        score = 0


        if "smooth" in key:
            score += 5

        if "stage" in key:
            score += 3

        if "norm" in key:
            score += 2

        if "water level" in key:
            score += 2


        candidates.append(
            (
                score,
                col
            )
        )


    if not candidates:

        raise ValueError(

            "Could not identify stage column.\n\n"

            f"Available columns:\n"
            f"{df.columns.tolist()}"
        )


    candidates.sort(
        reverse=True
    )


    return candidates[
        0
    ][1]


# ============================================================
# ALIGN PHASE + STAGE WITH EXACT WQ REFERENCE VECTOR
# ============================================================

def get_aligned_hydrologic_data(

    all_storms,

    classified_storms,

    storm_id,

    parameter,

    x_ref
):

    sid = storm_key(
        storm_id
    )


    storm_df = get_storm_dataframe(
        all_storms,
        sid
    )


    classified = classified_storms[
        sid
    ]


    parameter_col = find_parameter_column(
        storm_df,
        parameter
    )


    if parameter_col is None:

        raise ValueError(

            f"Could not identify {parameter} "
            f"column for Storm {sid}."
        )


    wq_full = pd.to_numeric(
        storm_df[
            parameter_col
        ],
        errors="coerce"
    )


    valid_wq = (
        wq_full.notna()
    )


    wq = (
        wq_full.loc[
            valid_wq
        ]
    )


    # --------------------------------------------------------
    # Confirm same reference vector
    # --------------------------------------------------------

    if len(wq) != len(x_ref):

        raise ValueError(

            f"WQ/reference length mismatch for "
            f"{parameter}, Storm {sid}: "

            f"{len(wq)} vs {len(x_ref)}"
        )


    if not np.allclose(

        wq.to_numpy(
            dtype=float
        ),

        np.asarray(
            x_ref,
            dtype=float
        ),

        rtol=1e-8,

        atol=1e-10

    ):

        raise ValueError(

            f"Reference mismatch for "
            f"{parameter}, Storm {sid}."
        )


    phase_col = find_phase_column(
        classified
    )


    stage_col = find_stage_column(
        classified
    )


    # --------------------------------------------------------
    # Direct index alignment
    # --------------------------------------------------------

    if wq.index.isin(
        classified.index
    ).all():


        phases = (

            classified[
                phase_col
            ]

            .reindex(
                wq.index
            )

            .to_numpy()
        )


        stage = (

            pd.to_numeric(

                classified[
                    stage_col
                ],

                errors="coerce"
            )

            .reindex(
                wq.index
            )

            .to_numpy(
                dtype=float
            )
        )


    # --------------------------------------------------------
    # Positional fallback
    # --------------------------------------------------------

    elif len(classified) == len(
        storm_df
    ):


        positions = np.flatnonzero(

            valid_wq.to_numpy()
        )


        phases = (

            classified[
                phase_col
            ]

            .iloc[
                positions
            ]

            .to_numpy()
        )


        stage = (

            pd.to_numeric(

                classified[
                    stage_col
                ],

                errors="coerce"
            )

            .iloc[
                positions
            ]

            .to_numpy(
                dtype=float
            )
        )


    else:

        raise ValueError(

            f"Cannot safely align hydrologic data for "
            f"{parameter}, Storm {sid}."
        )


    phases = np.array(

        [

            standardize_phase(
                p
            )

            for p in phases
        ],

        dtype=object
    )


    return (

        phases,

        stage,

        phase_col,

        stage_col
    )


# ============================================================
# NEW ALGORITHM 1:
# LOCAL MOVING MEAN
# ============================================================

def local_moving_mean_gapfill(

    x,

    n_each_side=2
):

    x = np.asarray(
        x,
        dtype=float
    )


    y = x.copy()


    observed_idx = np.flatnonzero(

        np.isfinite(
            x
        )
    )


    missing_idx = np.flatnonzero(

        ~np.isfinite(
            x
        )
    )


    if len(
        observed_idx
    ) == 0:

        return y


    for idx in missing_idx:


        left = observed_idx[

            observed_idx
            <
            idx

        ][
            -n_each_side:
        ]


        right = observed_idx[

            observed_idx
            >
            idx

        ][
            :n_each_side
        ]


        neighbors = np.concatenate(
            [
                left,
                right
            ]
        )


        if len(
            neighbors
        ) > 0:


            y[idx] = np.mean(

                x[
                    neighbors
                ]
            )


        else:


            y[idx] = np.nanmean(
                x
            )


    return y


# ============================================================
# ORIGINAL LINEAR FUNCTION
# Used by ΔH + HPWI as fallback
# ============================================================

def get_original_linear_function(
    scenario_name
):

    pool = choose_algorithm_pool(
        scenario_name
    )


    candidates = [

        "Linear_Interpolation",

        "Linear Interpolation",

        "Linear"
    ]


    for name in candidates:

        if name in pool:

            return pool[
                name
            ]


    raise KeyError(

        "Original Linear Interpolation function "
        "could not be identified.\n\n"

        f"Available algorithms:\n"
        f"{list(pool.keys())}"
    )


# ============================================================
# NEW ALGORITHM 2:
# DIRECT DELTA-H WEIGHTED INTERPOLATION
# ============================================================

def delta_h_weighted_gapfill(

    x,

    stage,

    linear_function
):

    x = np.asarray(
        x,
        dtype=float
    )


    stage = np.asarray(
        stage,
        dtype=float
    )


    if len(x) != len(stage):

        raise ValueError(

            "WQ and stage vectors "
            "have different lengths."
        )


    # Linear interpolation baseline/fallback
    y = np.asarray(

        match_length(

            linear_function(
                x
            ),

            len(x)
        ),

        dtype=float
    )


    observed_mask = np.isfinite(
        x
    )


    y[
        observed_mask
    ] = x[
        observed_mask
    ]


    missing_idx = np.flatnonzero(

        ~observed_mask
    )


    if len(
        missing_idx
    ) == 0:

        return y


    split_points = (

        np.where(

            np.diff(
                missing_idx
            )
            >
            1

        )[0]

        + 1
    )


    gap_groups = np.split(

        missing_idx,

        split_points
    )


    finite_stage = stage[

        np.isfinite(
            stage
        )
    ]


    stage_scale = (

        np.nanstd(
            finite_stage
        )

        if len(
            finite_stage
        ) > 1

        else 1.0
    )


    eps = max(

        1e-12,

        1e-8
        *
        max(
            stage_scale,
            1.0
        )
    )


    for gap in gap_groups:


        if len(
            gap
        ) == 0:

            continue


        left = (
            gap[0]
            - 1
        )


        right = (
            gap[-1]
            + 1
        )


        if (

            left < 0

            or

            right >= len(x)

        ):

            continue


        if (

            not np.isfinite(
                x[left]
            )

            or

            not np.isfinite(
                x[right]
            )

        ):

            continue


        H_left = stage[
            left
        ]


        H_right = stage[
            right
        ]


        if (

            not np.isfinite(
                H_left
            )

            or

            not np.isfinite(
                H_right
            )

        ):

            continue


        delta_H = (

            H_right
            -
            H_left
        )


        if abs(
            delta_H
        ) <= eps:

            continue


        H_gap = stage[
            gap
        ]


        alpha = (

            H_gap
            -
            H_left

        ) / delta_H


        usable = (

            np.isfinite(
                alpha
            )

            &

            (
                alpha >= 0
            )

            &

            (
                alpha <= 1
            )
        )


        if not np.any(
            usable
        ):

            continue


        usable_idx = gap[
            usable
        ]


        usable_alpha = alpha[
            usable
        ]


        y[
            usable_idx
        ] = (

            x[left]

            +

            usable_alpha

            *
            (
                x[right]
                -
                x[left]
            )
        )


    return y


# ============================================================
# NEW ALGORITHM 3:
# HYDROGRAPH-PATH WEIGHTED INTERPOLATION
# HPWI
# ============================================================

def hpwi_gapfill(

    x,

    stage,

    linear_function
):

    x = np.asarray(
        x,
        dtype=float
    )


    stage = np.asarray(
        stage,
        dtype=float
    )


    if len(x) != len(stage):

        raise ValueError(

            "WQ and stage vectors "
            "have different lengths."
        )


    y = np.asarray(

        match_length(

            linear_function(
                x
            ),

            len(x)
        ),

        dtype=float
    )


    observed_mask = np.isfinite(
        x
    )


    y[
        observed_mask
    ] = x[
        observed_mask
    ]


    missing_idx = np.flatnonzero(

        ~observed_mask
    )


    if len(
        missing_idx
    ) == 0:

        return y


    split_points = (

        np.where(

            np.diff(
                missing_idx
            )
            >
            1

        )[0]

        + 1
    )


    gap_groups = np.split(

        missing_idx,

        split_points
    )


    finite_stage = stage[

        np.isfinite(
            stage
        )
    ]


    stage_scale = (

        np.nanstd(
            finite_stage
        )

        if len(
            finite_stage
        ) > 1

        else 1.0
    )


    eps = max(

        1e-12,

        1e-8
        *
        max(
            stage_scale,
            1.0
        )
    )


    for gap in gap_groups:


        if len(
            gap
        ) == 0:

            continue


        left = (
            gap[0]
            - 1
        )


        right = (
            gap[-1]
            + 1
        )


        if (

            left < 0

            or

            right >= len(x)

        ):

            continue


        if (

            not np.isfinite(
                x[left]
            )

            or

            not np.isfinite(
                x[right]
            )

        ):

            continue


        H_segment = stage[

            left:
            right + 1
        ]


        if not np.all(

            np.isfinite(
                H_segment
            )
        ):

            continue


        # Absolute hydrograph movement
        dH = np.abs(

            np.diff(
                H_segment
            )
        )


        total_path = np.sum(
            dH
        )


        if (

            not np.isfinite(
                total_path
            )

            or

            total_path <= eps

        ):

            continue


        cumulative_path = np.cumsum(
            dH
        )


        local_positions = (

            gap
            -
            left
            -
            1
        )


        D_to_missing = cumulative_path[

            local_positions
        ]


        alpha = (

            D_to_missing

            /

            total_path
        )


        alpha = np.clip(

            alpha,

            0.0,

            1.0
        )


        y[
            gap
        ] = (

            x[left]

            +

            alpha

            *
            (
                x[right]
                -
                x[left]
            )
        )


    return y


# ============================================================
# LOAD 8-ALGORITHM RAW RESULTS
# Used only to recover the exact original trial seeds
# ============================================================

raw = pd.read_csv(
    results_file
)


for col in [

    "parameter",

    "scenario",

    "severity",

    "algorithm",

    "evaluation_domain",

    "status"

]:

    if col in raw.columns:

        raw[col] = (

            raw[col]

            .astype(
                str
            )
        )


# ============================================================
# RECOVER EXACT ORIGINAL DEGRADATION TRIALS
# ============================================================

trials = raw[

    raw[
        "scenario"
    ]
    ==
    scenario

].copy()


if "status" in trials.columns:


    ok = (

        trials[
            "status"
        ]

        .str.lower()

        .isin(
            [
                "ok",
                "success",
                "successful",
                "completed"
            ]
        )
    )


    if ok.any():

        trials = trials[
            ok
        ].copy()


trial_cols = [

    "parameter",

    "storm_id",

    "severity",

    "repetition",

    "seed"
]


trials = (

    trials[
        trial_cols
    ]

    .dropna(
        subset=[
            "seed"
        ]
    )

    .drop_duplicates()

    .sort_values(
        [
            "parameter",
            "storm_id",
            "severity",
            "repetition"
        ]
    )

    .reset_index(
        drop=True
    )
)


trials[
    "seed"
] = trials[
    "seed"
].astype(
    int
)


trials[
    "repetition"
] = trials[
    "repetition"
].astype(
    int
)


# ============================================================
# TRIAL COUNT VALIDATION
# ============================================================

print(
    "\n========================================"
)

print(
    "TRIAL COUNTS"
)

print(
    "========================================"
)


print(

    trials

    .groupby(
        "parameter"
    )

    .size()
)


expected_trials = 960


print(
    "\nTotal unique trials:",
    len(trials)
)


print(
    "Expected:",
    expected_trials
)


if len(
    trials
) != expected_trials:

    raise ValueError(

        f"Expected {expected_trials} unique trials "
        f"for {scenario}, but found {len(trials)}."
    )


# ============================================================
# BUILD PHASE + STAGE CACHE
# ============================================================

phase_cache = {}

stage_cache = {}

phase_column_cache = {}

stage_column_cache = {}


unique_parameter_storms = (

    trials[
        [
            "parameter",
            "storm_id"
        ]
    ]

    .drop_duplicates()
)


print(
    "\nBuilding hydrologic alignment cache..."
)


for _, row in unique_parameter_storms.iterrows():


    parameter = row[
        "parameter"
    ]


    sid = storm_key(
        row[
            "storm_id"
        ]
    )


    x_ref = np.asarray(

        get_reference_signal(

            all_storms=
                all_storms,

            storm_id=
                sid,

            parameter=
                parameter,

            dropna=
                True
        ),

        dtype=float
    )


    (

        phases,

        stage,

        phase_col,

        stage_col

    ) = get_aligned_hydrologic_data(

        all_storms=
            all_storms,

        classified_storms=
            classified_storms,

        storm_id=
            sid,

        parameter=
            parameter,

        x_ref=
            x_ref
    )


    phase_cache[
        (
            parameter,
            sid
        )
    ] = phases


    stage_cache[
        (
            parameter,
            sid
        )
    ] = stage


    phase_column_cache[
        (
            parameter,
            sid
        )
    ] = phase_col


    stage_column_cache[
        (
            parameter,
            sid
        )
    ] = stage_col


# ============================================================
# ALIGNMENT REPORT
# ============================================================

alignment_report = pd.DataFrame(

    [

        {

            "parameter":
                key[0],

            "storm_id":
                key[1],

            "phase_column":
                phase_column_cache[
                    key
                ],

            "stage_column":
                stage_column_cache[
                    key
                ],

            "finite_stage_percent":
                100
                *
                np.isfinite(
                    stage_cache[
                        key
                    ]
                ).mean()

        }

        for key in phase_cache.keys()
    ]
)


print(
    "\nHydrologic alignment report:"
)


print(

    alignment_report

    .sort_values(
        [
            "parameter",
            "storm_id"
        ]
    )

    .to_string(
        index=False
    )
)


# ============================================================
# VERIFY PHASE LABELS BEFORE RUNNING
# ============================================================

phase_check_rows = []


for (

    parameter,

    sid

), phases in phase_cache.items():


    values, counts = np.unique(

        phases,

        return_counts=True
    )


    for phase, n in zip(

        values,

        counts
    ):


        phase_check_rows.append({

            "parameter":
                parameter,

            "storm_id":
                sid,

            "phase":
                phase,

            "n":
                n
        })


phase_check = pd.DataFrame(
    phase_check_rows
)


print(
    "\nHydrologic phases found:"
)


print(

    phase_check

    .pivot_table(

        index=[
            "parameter",
            "storm_id"
        ],

        columns=
            "phase",

        values=
            "n",

        fill_value=
            0
    )
)


unexpected_phases = (

    set(
        phase_check[
            "phase"
        ].unique()
    )

    -
    set(
        phase_order
    )
)


if unexpected_phases:

    raise ValueError(

        "Unexpected hydrologic phase labels found:\n"

        + ", ".join(
            map(
                str,
                unexpected_phases
            )
        )
    )


# ============================================================
# ORIGINAL FIVE ALGORITHM POOL
# ============================================================

original_pool = choose_algorithm_pool(
    scenario
)


print(
    "\nOriginal algorithm pool:"
)


print(
    list(
        original_pool.keys()
    )
)


# ============================================================
# RESOLVE ORIGINAL ALGORITHM NAMES
# ============================================================

def resolve_original_algorithm(
    desired_name
):

    candidates = {

        "Linear_Interpolation": [

            "Linear_Interpolation",

            "Linear Interpolation",

            "Linear"
        ],

        "PCHIP": [

            "PCHIP"
        ],

        "Cubic_Spline": [

            "Cubic_Spline",

            "Cubic_Spline_Interpolation"
        ],

        "Kalman_Filter": [

            "Kalman_Filter",

            "Local_Level_Kalman_Smoother"
        ],

        "SSA_Gap_Filling": [

            "SSA_Gap_Filling"
        ]
    }


    if desired_name not in candidates:

        raise KeyError(

            f"No original-algorithm mapping "
            f"defined for {desired_name}."
        )


    for candidate in candidates[
        desired_name
    ]:


        if candidate in original_pool:

            return original_pool[
                candidate
            ]


    raise KeyError(

        f"Could not locate {desired_name}.\n"

        f"Available algorithms:\n"

        f"{list(original_pool.keys())}"
    )


# ============================================================
# PRE-CHECK ORIGINAL FIVE
# ============================================================

for algorithm in [

    "Linear_Interpolation",

    "PCHIP",

    "Cubic_Spline",

    "Kalman_Filter",

    "SSA_Gap_Filling"

]:

    resolve_original_algorithm(
        algorithm
    )


print(
    "\nOriginal five algorithm functions successfully resolved."
)


# ============================================================
# RECONSTRUCT EVERY ARTIFICIALLY MISSING POINT
# ============================================================

pointwise_rows = []


success_counter = {

    algorithm: 0

    for algorithm
    in algorithm_order
}


error_counter = {

    algorithm: 0

    for algorithm
    in algorithm_order
}


for count, trial in trials.iterrows():


    parameter = trial[
        "parameter"
    ]


    storm_id = storm_key(
        trial[
            "storm_id"
        ]
    )


    severity = trial[
        "severity"
    ]


    repetition = int(
        trial[
            "repetition"
        ]
    )


    seed = int(
        trial[
            "seed"
        ]
    )


    # --------------------------------------------------------
    # Reference signal
    # --------------------------------------------------------

    x_ref = np.asarray(

        get_reference_signal(

            all_storms=
                all_storms,

            storm_id=
                storm_id,

            parameter=
                parameter,

            dropna=
                True
        ),

        dtype=float
    )


    phases = phase_cache[
        (
            parameter,
            storm_id
        )
    ]


    stage = stage_cache[
        (
            parameter,
            storm_id
        )
    ]


    # --------------------------------------------------------
    # Reproduce exact original degradation
    # --------------------------------------------------------

    rng = np.random.default_rng(
        seed
    )


    x_deg, corruption_mask = (

        DEGRADATION_FUNCTIONS[
            scenario
        ](

            x_ref,

            severity,

            rng
        )
    )


    x_deg = np.asarray(
        x_deg,
        dtype=float
    )


    corruption_mask = np.asarray(
        corruption_mask,
        dtype=bool
    )


    # ========================================================
    # RUN ALL 8 ALGORITHMS
    # ========================================================

    for algorithm in algorithm_order:


        try:


            # =================================================
            # 1. LINEAR
            # =================================================

            if algorithm == "Linear_Interpolation":


                algorithm_fn = (
                    resolve_original_algorithm(
                        algorithm
                    )
                )


                x_hat = algorithm_fn(
                    x_deg
                )


            # =================================================
            # 2. PCHIP
            # =================================================

            elif algorithm == "PCHIP":


                algorithm_fn = (
                    resolve_original_algorithm(
                        algorithm
                    )
                )


                x_hat = algorithm_fn(
                    x_deg
                )


            # =================================================
            # 3. CUBIC SPLINE
            # =================================================

            elif algorithm == "Cubic_Spline":


                algorithm_fn = (
                    resolve_original_algorithm(
                        algorithm
                    )
                )


                x_hat = algorithm_fn(
                    x_deg
                )


            # =================================================
            # 4. LOCAL MOVING MEAN
            # =================================================

            elif algorithm == "Local_Moving_Mean":


                x_hat = local_moving_mean_gapfill(

                    x=
                        x_deg,

                    n_each_side=
                        2
                )


            # =================================================
            # 5. DELTA-H WEIGHTED
            # =================================================

            elif (
                algorithm
                ==
                "Delta_H_Weighted_Interpolation"
            ):


                linear_fn = (
                    get_original_linear_function(
                        scenario
                    )
                )


                x_hat = (
                    delta_h_weighted_gapfill(

                        x=
                            x_deg,

                        stage=
                            stage,

                        linear_function=
                            linear_fn
                    )
                )


            # =================================================
            # 6. HPWI
            # =================================================

            elif algorithm == "HPWI":


                linear_fn = (
                    get_original_linear_function(
                        scenario
                    )
                )


                x_hat = hpwi_gapfill(

                    x=
                        x_deg,

                    stage=
                        stage,

                    linear_function=
                        linear_fn
                )


            # =================================================
            # 7. KALMAN
            # =================================================

            elif algorithm == "Kalman_Filter":


                algorithm_fn = (
                    resolve_original_algorithm(
                        algorithm
                    )
                )


                x_hat = algorithm_fn(
                    x_deg
                )


            # =================================================
            # 8. SSA GAP
            # =================================================

            elif algorithm == "SSA_Gap_Filling":


                algorithm_fn = (
                    resolve_original_algorithm(
                        algorithm
                    )
                )


                x_hat = algorithm_fn(
                    x_deg
                )


            else:

                raise KeyError(

                    f"Unknown algorithm: "
                    f"{algorithm}"
                )


            # -------------------------------------------------
            # Ensure correct length
            # -------------------------------------------------

            x_hat = np.asarray(

                match_length(

                    x_hat,

                    len(
                        x_ref
                    )
                ),

                dtype=float
            )


            # -------------------------------------------------
            # Only artificially missing observations
            # -------------------------------------------------

            valid = (

                corruption_mask

                &

                np.isfinite(
                    x_ref
                )

                &

                np.isfinite(
                    x_hat
                )
            )


            missing_idx = np.where(
                valid
            )[0]


            if len(
                missing_idx
            ) == 0:

                continue


            temp = pd.DataFrame({

                "parameter":
                    parameter,

                "storm_id":
                    storm_id,

                "severity":
                    severity,

                "repetition":
                    repetition,

                "seed":
                    seed,

                "algorithm":
                    algorithm,

                "algorithm_display":
                    algo_display.get(
                        algorithm,
                        algorithm
                    ),

                "point_index":
                    missing_idx,

                "reference":
                    x_ref[
                        missing_idx
                    ],

                "reconstructed":
                    x_hat[
                        missing_idx
                    ],

                "hydrologic_phase":
                    phases[
                        missing_idx
                    ]
            })


            pointwise_rows.append(
                temp
            )


            success_counter[
                algorithm
            ] += 1


        except Exception as exc:


            error_counter[
                algorithm
            ] += 1


            # Print only first 5 errors / algorithm
            if (
                error_counter[
                    algorithm
                ]
                <=
                5
            ):


                print(

                    f"\nERROR — "
                    f"{algorithm}"

                    f"\nParameter: "
                    f"{parameter}"

                    f"\nStorm: "
                    f"{storm_id}"

                    f"\nSeverity: "
                    f"{severity}"

                    f"\nRepetition: "
                    f"{repetition}"

                    f"\nSeed: "
                    f"{seed}"

                    f"\nMessage: "
                    f"{exc}\n"
                )


    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    if (
        count + 1
    ) % 100 == 0:


        print(

            f"Processed "
            f"{count + 1}/"
            f"{len(trials)} trials"
        )


# ============================================================
# EXECUTION SUMMARY
# ============================================================

print(
    "\n========================================"
)

print(
    "ALGORITHM EXECUTION SUMMARY"
)

print(
    "========================================"
)


for algorithm in algorithm_order:


    print(

        f"{algo_display.get(algorithm, algorithm):15s} "

        f"success = "
        f"{success_counter[algorithm]:4d}"

        f" | errors = "
        f"{error_counter[algorithm]:4d}"
    )


# ============================================================
# BUILD POINTWISE DATASET
# ============================================================

if not pointwise_rows:

    raise RuntimeError(

        "No pointwise results were generated."
    )


pointwise = pd.concat(

    pointwise_rows,

    ignore_index=True
)


# ============================================================
# VERIFY ALL 8 ARE PRESENT BEFORE CONTINUING
# ============================================================

present_algorithms = sorted(

    pointwise[
        "algorithm"
    ]

    .unique()
)


print(
    "\nAlgorithms actually present:"
)


for i, algorithm in enumerate(

    present_algorithms,

    start=1
):

    print(

        f"{i}. "
        f"{algo_display.get(algorithm, algorithm)} "
        f"[{algorithm}]"
    )


print(
    "\nNumber of algorithms present:",
    len(
        present_algorithms
    )
)


missing_algorithms = [

    algorithm

    for algorithm in algorithm_order

    if algorithm not in present_algorithms
]


if missing_algorithms:


    print(
        "\nMISSING ALGORITHMS:"
    )


    for algorithm in missing_algorithms:

        print(
            " - ",
            algorithm
        )


    raise RuntimeError(

        "Pointwise reconstruction contains fewer than "
        "8 algorithms. See execution errors above."
    )


if len(
    present_algorithms
) != 8:

    raise RuntimeError(

        f"Expected 8 algorithms but found "
        f"{len(present_algorithms)}."
    )


print(
    "\nPASS: all 8 algorithms successfully entered "
    "the pointwise dataset."
)


# ============================================================
# ERROR COLUMNS
# ============================================================

pointwise[
    "Error"
] = (

    pointwise[
        "reconstructed"
    ]

    -

    pointwise[
        "reference"
    ]
)


pointwise[
    "Absolute_Error"
] = np.abs(

    pointwise[
        "Error"
    ]
)


pointwise[
    "Squared_Error"
] = (

    pointwise[
        "Error"
    ]
    **
    2
)


# ============================================================
# KEEP ONLY FINAL HYDROLOGIC CLASSES
# ============================================================

pointwise = pointwise[

    pointwise[
        "hydrologic_phase"
    ]

    .isin(
        phase_order
    )

].copy()


# ============================================================
# SAVE FULL POINTWISE DATA
# ============================================================

pointwise_file = (

    output_dir

    /

    f"{scenario}_pointwise_reconstruction_8alg.csv"
)


pointwise.to_csv(

    pointwise_file,

    index=False
)


print(
    "\nAlgorithm-specific reconstructed observations:"
)

print(
    f"{len(pointwise):,}"
)


# ============================================================
# UNIQUE MISSING OBSERVATIONS
# Remove 8-fold algorithm duplication
# ============================================================

unique_missing = (

    pointwise

    .drop_duplicates(

        [

            "parameter",

            "storm_id",

            "severity",

            "repetition",

            "seed",

            "point_index"
        ]
    )
)


print(
    "\nUnique artificially missing observations:"
)

print(
    f"{len(unique_missing):,}"
)


coverage = (

    len(
        pointwise
    )

    /

    len(
        unique_missing
    )
)


print(
    "\nAverage algorithm coverage per missing point:"
)

print(
    f"{coverage:.2f}"
)


if not np.isclose(
    coverage,
    8.0,
    atol=0.01
):

    print(

        "\nWARNING: coverage is not exactly 8.00."
    )


# ============================================================
# HYDROLOGIC-PHASE DISTRIBUTION
# ============================================================

phase_distribution = (

    unique_missing

    .groupby(

        [
            "parameter",
            "hydrologic_phase"
        ]
    )

    .size()

    .rename(
        "n"
    )

    .reset_index()
)


phase_distribution[
    "Percent"
] = (

    phase_distribution[
        "n"
    ]

    /

    phase_distribution

    .groupby(
        "parameter"
    )[
        "n"
    ]

    .transform(
        "sum"
    )

    *
    100
)


phase_distribution.to_csv(

    output_dir

    /

    f"{scenario}_hydrologic_phase_distribution.csv",

    index=False
)


print(
    "\nDistribution of artificially missing observations "
    "by hydrologic phase:"
)


print(

    phase_distribution

    .pivot(

        index=
            "hydrologic_phase",

        columns=
            "parameter",

        values=
            "Percent"
    )

    .reindex(
        phase_order
    )

    .round(
        1
    )
)


# ============================================================
# PERFORMANCE BY HYDROLOGIC PHASE
# ============================================================

summary = (

    pointwise

    .groupby(

        [

            "parameter",

            "algorithm",

            "algorithm_display",

            "hydrologic_phase"
        ],

        as_index=False
    )

    .agg(

        n=(
            "Absolute_Error",
            "size"
        ),

        MAE=(
            "Absolute_Error",
            "mean"
        ),

        RMSE=(

            "Squared_Error",

            lambda x:
                np.sqrt(
                    np.mean(
                        x
                    )
                )
        ),

        Median_AE=(
            "Absolute_Error",
            "median"
        ),

        Bias=(
            "Error",
            "mean"
        )
    )
)


summary.to_csv(

    output_dir

    /

    f"{scenario}_performance_by_hydrologic_phase_8alg.csv",

    index=False
)


# ============================================================
# SEVERITY-SPECIFIC SUMMARY
# ============================================================

summary_severity = (

    pointwise

    .groupby(

        [

            "parameter",

            "severity",

            "algorithm",

            "algorithm_display",

            "hydrologic_phase"
        ],

        as_index=False
    )

    .agg(

        n=(
            "Absolute_Error",
            "size"
        ),

        MAE=(
            "Absolute_Error",
            "mean"
        ),

        RMSE=(

            "Squared_Error",

            lambda x:
                np.sqrt(
                    np.mean(
                        x
                    )
                )
        ),

        Median_AE=(
            "Absolute_Error",
            "median"
        ),

        Bias=(
            "Error",
            "mean"
        )
    )
)


summary_severity.to_csv(

    output_dir

    /

    f"{scenario}_performance_by_phase_and_severity_8alg.csv",

    index=False
)


# ============================================================
# QUICK MAE TABLE
# ============================================================

print(
    "\n========================================"
)

print(
    "MAE BY HYDROLOGIC PHASE"
)

print(
    "========================================"
)


mae_table = (

    summary

    .pivot_table(

        index=[
            "parameter",
            "algorithm_display"
        ],

        columns=
            "hydrologic_phase",

        values=
            "MAE"
    )

    .reindex(
        columns=
            phase_order
    )

    .round(
        3
    )
)


print(
    mae_table
)


# ============================================================
# OPTIONAL:
# SPECIAL LINEAR vs DELTA-H vs HPWI PHASE COMPARISON
# ============================================================

hydrology_compare = (

    summary[

        summary[
            "algorithm"
        ]

        .isin(

            [

                "Linear_Interpolation",

                "Delta_H_Weighted_Interpolation",

                "HPWI"
            ]
        )
    ]

    .copy()
)


hydrology_compare.to_csv(

    output_dir

    /

    f"{scenario}_Linear_DeltaH_HPWI_by_phase.csv",

    index=False
)


print(
    "\nLinear vs ΔH weighted vs HPWI by phase:"
)


print(

    hydrology_compare[

        [

            "parameter",

            "algorithm_display",

            "hydrologic_phase",

            "MAE",

            "RMSE"
        ]
    ]

    .sort_values(

        [

            "parameter",

            "hydrologic_phase",

            "algorithm_display"
        ]
    )

    .round(
        3
    )

    .to_string(
        index=False
    )
)


# ============================================================
# PARITY PLOT SETTINGS
# ============================================================

plt.rcParams.update({

    "font.family":
        "serif",

    "font.serif":
        [
            "Times New Roman",
            "Times",
            "DejaVu Serif"
        ],

    "font.size":
        11,

    "pdf.fonttype":
        42,

    "ps.fonttype":
        42,

    "figure.facecolor":
        "white",

    "axes.facecolor":
        "white",

    "savefig.facecolor":
        "white"
})


# ============================================================
# COLORS
# ============================================================

cmap = plt.get_cmap(
    "tab10"
)


phase_colors = {

    "Baseline/inter-event":
        cmap(0),

    "Storm limb":
        cmap(1),

    "Peak window":
        cmap(2)
}


# ============================================================
# VISUAL SUBSAMPLING ONLY
#
# ALL observations remain in quantitative statistics.
# This only prevents the scatter plots from becoming too dense.
# ============================================================

MAX_POINTS_PER_PHASE = 1500


def subsample_for_plot(

    df,

    max_n=
        MAX_POINTS_PER_PHASE,

    seed=
        12345
):

    if len(
        df
    ) <= max_n:

        return df


    return df.sample(

        n=
            max_n,

        random_state=
            seed
    )


# ============================================================
# CREATE 4 × 8 PARITY MATRIX
# ============================================================

fig, axes = plt.subplots(

    len(
        parameter_order
    ),

    len(
        algorithm_order
    ),

    figsize=(
        26,
        12
    ),

    facecolor=
        "white"
)


# ============================================================
# PLOT
# ============================================================

for row, parameter in enumerate(
    parameter_order
):


    parameter_data = pointwise[

        pointwise[
            "parameter"
        ]

        ==
        parameter
    ]


    # --------------------------------------------------------
    # Common limits for every algorithm within same parameter
    # --------------------------------------------------------

    all_values = np.concatenate(

        [

            parameter_data[
                "reference"
            ]
            .to_numpy(),

            parameter_data[
                "reconstructed"
            ]
            .to_numpy()
        ]
    )


    all_values = all_values[

        np.isfinite(
            all_values
        )
    ]


    lo = np.min(
        all_values
    )


    hi = np.max(
        all_values
    )


    padding = (

        0.04
        *
        (
            hi
            -
            lo
        )

        if hi > lo

        else 1
    )


    lo -= padding

    hi += padding


    # ========================================================
    # EACH ALGORITHM
    # ========================================================

    for col, algorithm in enumerate(
        algorithm_order
    ):


        ax = axes[
            row,
            col
        ]


        data = parameter_data[

            parameter_data[
                "algorithm"
            ]

            ==
            algorithm
        ]


        # ----------------------------------------------------
        # Scatter by phase
        # ----------------------------------------------------

        for phase_index, phase in enumerate(
            phase_order
        ):


            d = data[

                data[
                    "hydrologic_phase"
                ]

                ==
                phase
            ]


            if d.empty:

                continue


            d_plot = subsample_for_plot(

                d,

                max_n=
                    MAX_POINTS_PER_PHASE,

                seed=
                    (
                        1000
                        +
                        row
                        *
                        100
                        +
                        col
                        *
                        10
                        +
                        phase_index
                    )
            )


            ax.scatter(

                d_plot[
                    "reference"
                ],

                d_plot[
                    "reconstructed"
                ],

                s=
                    7,

                alpha=
                    0.24,

                color=
                    phase_colors[
                        phase
                    ],

                edgecolors=
                    "none",

                rasterized=
                    True,

                zorder=
                    2
            )


        # ----------------------------------------------------
        # 1:1 line
        # ----------------------------------------------------

        ax.plot(

            [
                lo,
                hi
            ],

            [
                lo,
                hi
            ],

            "--",

            color=
                "black",

            linewidth=
                1.1,

            zorder=
                5
        )


        # ----------------------------------------------------
        # Regression based on ALL points
        # ----------------------------------------------------

        valid = (

            data[

                [
                    "reference",
                    "reconstructed"
                ]
            ]

            .dropna()
        )


        if len(
            valid
        ) > 2:


            x = valid[
                "reference"
            ].to_numpy()


            y = valid[
                "reconstructed"
            ].to_numpy()


            slope, intercept = np.polyfit(

                x,

                y,

                1
            )


            xx = np.linspace(

                lo,

                hi,

                100
            )


            yy = (

                slope
                *
                xx

                +
                intercept
            )


            ax.plot(

                xx,

                yy,

                color=
                    "gray",

                linewidth=
                    1.1,

                zorder=
                    4
            )


            # ------------------------------------------------
            # Statistics using ALL points
            # ------------------------------------------------

            r = np.corrcoef(
                x,
                y
            )[0, 1]


            r2 = r ** 2


            rmse = np.sqrt(

                np.mean(

                    (
                        y
                        -
                        x
                    )
                    **
                    2
                )
            )


            mae = np.mean(

                np.abs(

                    y
                    -
                    x
                )
            )


            ax.text(

                0.04,

                0.96,

                (
                    f"$R^2$={r2:.2f}\n"
                    f"RMSE={rmse:.2f}\n"
                    f"MAE={mae:.2f}"
                ),

                transform=
                    ax.transAxes,

                va=
                    "top",

                fontsize=
                    7.2,

                zorder=
                    6
            )


        # ----------------------------------------------------
        # Common limits
        # ----------------------------------------------------

        ax.set_xlim(
            lo,
            hi
        )


        ax.set_ylim(
            lo,
            hi
        )


        # ----------------------------------------------------
        # Algorithm titles
        # ----------------------------------------------------

        if row == 0:


            ax.set_title(

                algo_display.get(
                    algorithm,
                    algorithm
                ),

                fontsize=
                    11,

                fontweight=
                    "bold",

                pad=
                    6
            )


        # ----------------------------------------------------
        # Parameter/y-axis
        # ----------------------------------------------------

        if col == 0:


            ax.set_ylabel(

                (
                    f"{parameter_display.get(parameter, parameter)}\n"
                    f"Reconstructed value"
                ),

                fontsize=
                    10
            )


        # ----------------------------------------------------
        # X-axis
        # ----------------------------------------------------

        if (

            row

            ==

            len(
                parameter_order
            )
            - 1

        ):


            ax.set_xlabel(

                "Reference value",

                fontsize=
                    9
            )


        ax.tick_params(

            labelsize=
                7
        )


# ============================================================
# SHARED LEGEND
# ============================================================

legend_handles = [

    Line2D(

        [0],

        [0],

        marker=
            "o",

        linestyle=
            "",

        markersize=
            7,

        markerfacecolor=
            phase_colors[
                phase
            ],

        markeredgecolor=
            "none",

        label=
            phase
    )

    for phase
    in phase_order
]


legend_handles += [

    Line2D(

        [0],

        [0],

        linestyle=
            "--",

        color=
            "black",

        linewidth=
            1.1,

        label=
            "1:1 line"
    ),


    Line2D(

        [0],

        [0],

        linestyle=
            "-",

        color=
            "gray",

        linewidth=
            1.1,

        label=
            "Linear fit"
    )
]


fig.legend(

    handles=
        legend_handles,

    loc=
        "lower center",

    ncol=
        5,

    frameon=
        False,

    fontsize=
        10,

    bbox_to_anchor=
        (
            0.5,
            0.012
        )
)


# ============================================================
# SPACING
# ============================================================

fig.subplots_adjust(

    left=
        0.065,

    right=
        0.995,

    top=
        0.95,

    bottom=
        0.09,

    wspace=
        0.16,

    hspace=
        0.25
)


# ============================================================
# SAVE FIGURE
# ============================================================

png_path = (

    output_dir

    /

    f"{scenario}_pointwise_parity_hydrologic_phase_8alg.png"
)


pdf_path = (

    output_dir

    /

    f"{scenario}_pointwise_parity_hydrologic_phase_8alg.pdf"
)


fig.savefig(

    png_path,

    dpi=
        600,

    bbox_inches=
        "tight",

    facecolor=
        "white"
)


fig.savefig(

    pdf_path,

    bbox_inches=
        "tight",

    facecolor=
        "white"
)


plt.show()


# ============================================================
# FINAL REPORT
# ============================================================

print(
    "\n========================================"
)

print(
    "ANALYSIS COMPLETE"
)

print(
    "========================================"
)


print(
    "\nScenario:",
    scenario
)


print(
    "\nNumber of algorithms:",
    pointwise[
        "algorithm"
    ].nunique()
)


print(
    "\nAlgorithms:"
)


for algorithm in algorithm_order:


    print(

        " -",

        algo_display.get(
            algorithm,
            algorithm
        )
    )


print(
    "\nPointwise CSV:"
)

print(
    pointwise_file.resolve()
)


print(
    "\nFigure PNG:"
)

print(
    png_path.resolve()
)


print(
    "\nFigure PDF:"
)

print(
    pdf_path.resolve()
)

In [ ]:
# ============================================================
# SPLIT PARITY FIGURE INTO TWO 4 × 4 FIGURES
#
# Figure A:
#   Linear, PCHIP, Spline, Moving mean
#
# Figure B:
#   ΔH weighted, HPWI, Kalman, SSA gap
# ============================================================

figure_groups = {

    "A": [
        "Linear_Interpolation",
        "PCHIP",
        "Cubic_Spline",
        "Local_Moving_Mean"
    ],

    "B": [
        "Delta_H_Weighted_Interpolation",
        "HPWI",
        "Kalman_Filter",
        "SSA_Gap_Filling"
    ]
}


# ============================================================
# COMMON ROBUST AXIS LIMITS BY PARAMETER
#
# Uses ALL 8 algorithms so Figure A and Figure B have
# identical scales for each water-quality parameter.
#
# Extreme 0.2% tails are excluded from the DISPLAY ONLY.
# All observations remain in MAE, RMSE, r², etc.
# ============================================================

parameter_limits = {}

for parameter in parameter_order:

    parameter_data = pointwise[
        pointwise["parameter"] == parameter
    ].copy()

    if parameter_data.empty:
        raise ValueError(
            f"No pointwise data found for parameter: {parameter}"
        )

    all_values = np.concatenate([
        parameter_data["reference"].to_numpy(dtype=float),
        parameter_data["reconstructed"].to_numpy(dtype=float)
    ])

    all_values = all_values[
        np.isfinite(all_values)
    ]

    if len(all_values) == 0:
        raise ValueError(
            f"No finite plotting values found for parameter: {parameter}"
        )

    # --------------------------------------------------------
    # Robust limits, consistent with original figure
    # --------------------------------------------------------

    lo = np.nanpercentile(
        all_values,
        0.2
    )

    hi = np.nanpercentile(
        all_values,
        99.8
    )

    padding = (
        0.04 * (hi - lo)
        if hi > lo
        else 1.0
    )

    lo = lo - padding
    hi = hi + padding

    # --------------------------------------------------------
    # Turbidity cannot be physically negative.
    # This changes DISPLAY ONLY, not predictions or statistics.
    # --------------------------------------------------------

    if parameter == "Turbidity":
        lo = 0.0

    parameter_limits[parameter] = (
        lo,
        hi
    )


# ============================================================
# VERIFY PARAMETER LIMITS BEFORE PLOTTING
# ============================================================

print("\nParameter axis limits:")

for parameter in parameter_order:

    print(
        f"{parameter:10s}: "
        f"{parameter_limits[parameter][0]:.3f} "
        f"to "
        f"{parameter_limits[parameter][1]:.3f}"
    )

print(
    "\nKeys in parameter_limits:",
    list(parameter_limits.keys())
)

# ============================================================
# SHARED LEGEND HANDLES
# ============================================================

legend_handles = [

    Line2D(
        [0],
        [0],
        marker="o",
        linestyle="",
        markersize=7,
        markerfacecolor=phase_colors[phase],
        markeredgecolor="none",
        label=phase
    )

    for phase in phase_order
]


legend_handles += [

    Line2D(
        [0],
        [0],
        linestyle="--",
        color="black",
        linewidth=1.1,
        label="1:1 line"
    ),

    Line2D(
        [0],
        [0],
        linestyle="-",
        color="gray",
        linewidth=1.1,
        label="Linear fit"
    )
]


# ============================================================
# CREATE EACH 4 × 4 FIGURE
# ============================================================

saved_figures = []


for group_name, group_algorithms in figure_groups.items():

    fig, axes = plt.subplots(
        nrows=4,
        ncols=4,
        figsize=(13.5, 12),
        facecolor="white"
    )


    # ========================================================
    # PARAMETER ROWS
    # ========================================================

    for row, parameter in enumerate(parameter_order):

        parameter_data = pointwise[
            pointwise["parameter"] == parameter
        ]

        lo, hi = parameter_limits[
            parameter
        ]


        # ====================================================
        # ALGORITHM COLUMNS
        # ====================================================

        for col, algorithm in enumerate(group_algorithms):

            ax = axes[
                row,
                col
            ]

            data = parameter_data[
                parameter_data["algorithm"] == algorithm
            ]


            # =================================================
            # SCATTER BY HYDROLOGIC PHASE
            # =================================================

            for phase_index, phase in enumerate(phase_order):

                d = data[
                    data["hydrologic_phase"] == phase
                ]

                if d.empty:
                    continue


                # Visual subsampling only
                d_plot = subsample_for_plot(
                    d,
                    max_n=MAX_POINTS_PER_PHASE,
                    seed=(
                        1000
                        + row * 100
                        + col * 10
                        + phase_index
                    )
                )


                ax.scatter(
                    d_plot["reference"],
                    d_plot["reconstructed"],
                    s=8,
                    alpha=0.25,
                    color=phase_colors[phase],
                    edgecolors="none",
                    rasterized=True,
                    zorder=2
                )


            # =================================================
            # 1:1 AGREEMENT LINE
            # =================================================

            ax.plot(
                [lo, hi],
                [lo, hi],
                "--",
                color="black",
                linewidth=1.1,
                zorder=5
            )


            # =================================================
            # LINEAR FIT + STATISTICS
            #
            # Statistics use ALL points, not subsampled points.
            # =================================================

            valid = (
                data[
                    [
                        "reference",
                        "reconstructed"
                    ]
                ]
                .dropna()
            )


            if len(valid) > 2:

                x = valid[
                    "reference"
                ].to_numpy()

                y = valid[
                    "reconstructed"
                ].to_numpy()


                slope, intercept = np.polyfit(
                    x,
                    y,
                    1
                )


                xx = np.linspace(
                    lo,
                    hi,
                    100
                )

                yy = (
                    slope * xx
                    + intercept
                )


                ax.plot(
                    xx,
                    yy,
                    color="gray",
                    linewidth=1.1,
                    zorder=4
                )


                r = np.corrcoef(
                    x,
                    y
                )[0, 1]

                r2 = r ** 2


                rmse = np.sqrt(
                    np.mean(
                        (y - x) ** 2
                    )
                )


                mae = np.mean(
                    np.abs(
                        y - x
                    )
                )


                ax.text(
                    0.04,
                    0.96,
                    (
                        f"$R^2$={r2:.2f}\n"
                        f"RMSE={rmse:.2f}\n"
                        f"MAE={mae:.2f}"
                    ),
                    transform=ax.transAxes,
                    va="top",
                    fontsize=8
                )


            # =================================================
            # COMMON AXES
            # =================================================

            ax.set_xlim(
                lo,
                hi
            )

            ax.set_ylim(
                lo,
                hi
            )


            # =================================================
            # ALGORITHM TITLE
            # =================================================

            if row == 0:

                ax.set_title(
                    algo_display.get(
                        algorithm,
                        algorithm
                    ),
                    fontsize=12,
                    fontweight="bold",
                    pad=7
                )


            # =================================================
            # Y AXIS
            # =================================================

            if col == 0:

                ax.set_ylabel(
                    (
                        f"{parameter_display.get(parameter, parameter)}\n"
                        "Reconstructed value"
                    ),
                    fontsize=10
                )


            # =================================================
            # X AXIS
            # =================================================

            if row == len(parameter_order) - 1:

                ax.set_xlabel(
                    "Reference value",
                    fontsize=10
                )


            ax.tick_params(
                labelsize=8
            )


    # ========================================================
    # SHARED LEGEND FOR EACH FIGURE
    # ========================================================

    fig.legend(
        handles=legend_handles,
        loc="lower center",
        ncol=5,
        frameon=False,
        fontsize=10,
        bbox_to_anchor=(0.5, 0.012)
    )


    # ========================================================
    # SPACING
    # ========================================================

    fig.subplots_adjust(
        left=0.09,
        right=0.99,
        top=0.95,
        bottom=0.095,
        wspace=0.22,
        hspace=0.25
    )


    # ========================================================
    # SAVE
    # ========================================================

    png_path = (
        output_dir
        /
        f"{scenario}_pointwise_parity_hydrologic_phase_"
        f"8alg_part_{group_name}.png"
    )


    pdf_path = (
        output_dir
        /
        f"{scenario}_pointwise_parity_hydrologic_phase_"
        f"8alg_part_{group_name}.pdf"
    )


    fig.savefig(
        png_path,
        dpi=600,
        bbox_inches="tight",
        facecolor="white"
    )


    fig.savefig(
        pdf_path,
        bbox_inches="tight",
        facecolor="white"
    )


    saved_figures.append(
        (
            group_name,
            png_path,
            pdf_path
        )
    )


    plt.show()


# ============================================================
# FINAL SAVED-FIGURE REPORT
# ============================================================

print(
    "\n========================================"
)

print(
    "4 × 4 PARITY FIGURES SAVED"
)

print(
    "========================================"
)


for group_name, png_path, pdf_path in saved_figures:

    print(
        f"\nFigure {group_name}:"
    )

    print(
        "PNG:",
        png_path.resolve()
    )

    print(
        "PDF:",
        pdf_path.resolve()
    )

## Sample Reconstruction (Figure S2, S4, S6 and S7)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ------------------------------------------------------------
# User choices
# ------------------------------------------------------------
storm_num = 4
storm_label = f"Storm_{storm_num}"
parameter = "DO"  # e.g. "Turbidity", "EC", "DO", "Temp"
scenario = "field_realistic_fault"   # random_missing, block_missing, additive_noise, field_realistic_fault
severity = "severe"
rep_seed = 44

# None = plot every algorithm available for the scenario
algorithm_subset = None

outdir = Path("analysis_outputs/sample_reconstructions")
outdir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Reference signal
# ------------------------------------------------------------
x_ref = get_reference_signal(all_storms, storm_num, parameter).astype(float)
t = np.arange(len(x_ref))

rng = np.random.default_rng(rep_seed)
x_deg, mask = DEGRADATION_FUNCTIONS[scenario](x_ref.copy(), severity, rng)

print("Signal length:", len(x_ref))
print("Scenario:", scenario)
print("Severity:", severity)
print("Parameter:", parameter)
print("Storm:", storm_label)
print("Number affected:", np.sum(mask) if mask is not None else "N/A")

# ------------------------------------------------------------
# Helper: extract prediction from functions that may also
# return diagnostics
# ------------------------------------------------------------
def extract_prediction(result):
    return result[0] if isinstance(result, tuple) else result

# ------------------------------------------------------------
# Get hydrologic series aligned EXACTLY with x_ref
# Only needed for missingness scenarios
# ------------------------------------------------------------
def get_aligned_hydro(all_storms, storm_num, parameter, x_ref):
    df = get_storm_dataframe(all_storms, storm_num).copy()

    pcol = find_parameter_column(df, parameter)
    if pcol is None:
        raise ValueError(f"Could not identify {parameter} column.")

    wq = pd.to_numeric(df[pcol], errors="coerce")
    valid = wq.notna()
    ref_index = df.index[valid]
    wq_check = wq.loc[valid].to_numpy(dtype=float)

    if len(wq_check) != len(x_ref):
        raise ValueError(f"WQ alignment mismatch: {len(wq_check)} vs {len(x_ref)}.")

    if not np.allclose(wq_check, x_ref, equal_nan=True):
        raise ValueError("Aligned WQ series does not match x_ref.")

    # Prefer Stage_smoothed because this matches the original
    # hydrology-informed experiment
    if "classified_storms" in globals():
        try:
            hdf = get_storm_dataframe(classified_storms, storm_num).copy()

            if "Stage_smoothed" in hdf.columns:
                stage = pd.to_numeric(hdf["Stage_smoothed"], errors="coerce")
                stage = stage.reindex(ref_index).to_numpy(dtype=float)

                if len(stage) == len(x_ref) and np.isfinite(stage).mean() > 0.95:
                    return stage, "Stage_smoothed"
        except Exception:
            pass

    # Fallback: simultaneously measured Radar from same storm
    if "Radar" not in df.columns:
        raise ValueError("Neither aligned Stage_smoothed nor Radar is available.")

    radar = pd.to_numeric(df["Radar"], errors="coerce").loc[valid].to_numpy(dtype=float)

    if len(radar) != len(x_ref):
        raise ValueError(f"Radar alignment mismatch: {len(radar)} vs {len(x_ref)}.")

    return radar, "Radar"

# ------------------------------------------------------------
# Choose algorithms
# ------------------------------------------------------------
algorithm_pool = choose_algorithm_pool(scenario).copy()

if scenario in ["random_missing", "block_missing"]:
    hydro, hydro_source = get_aligned_hydro(all_storms, storm_num, parameter, x_ref)
    print("Hydrologic auxiliary:", hydro_source)
    print("Hydrologic length:", len(hydro))
    print("Finite hydrologic values:", f"{100*np.isfinite(hydro).mean():.1f}%")

    linear_key = next((k for k in ["Linear_Interpolation", "Linear Interpolation", "Linear"] if k in algorithm_pool), None)
    if linear_key is None:
        raise KeyError(f"Linear interpolation not found. Available: {list(algorithm_pool)}")

    linear_fn = algorithm_pool[linear_key]

    algorithm_pool["Local_Moving_Mean"] = lambda x: local_moving_mean_gapfill(x=x, n_each_side=2)
    algorithm_pool["Delta_H_Weighted_Interpolation"] = lambda x: delta_h_weighted_gapfill(x=x, stage=hydro, linear_function=linear_fn)
    algorithm_pool["HPWI"] = lambda x: hpwi_gapfill(x=x, stage=hydro, linear_function=linear_fn)

if algorithm_subset is not None:
    algorithm_pool = {k: v for k, v in algorithm_pool.items() if k in algorithm_subset}

print("Algorithms to plot:", list(algorithm_pool.keys()))

# ------------------------------------------------------------
# Run reconstruction / correction
# ------------------------------------------------------------
reconstructions = {}

for alg_name, alg_func in algorithm_pool.items():
    try:
        result = alg_func(x_deg.copy())
        x_hat = np.asarray(extract_prediction(result), dtype=float)
        x_hat = x_hat[:min(len(x_ref), len(x_hat))]
        reconstructions[alg_name] = x_hat
    except Exception as e:
        print(f"{alg_name} failed: {e}")
        reconstructions[alg_name] = None

# ------------------------------------------------------------
# Plot all algorithms only
# ------------------------------------------------------------
valid_algs = [k for k, v in reconstructions.items() if v is not None]
n_algs = len(valid_algs)

ncols = 3
nrows = int(np.ceil(n_algs / ncols))

fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(16, 4 * nrows),
    facecolor="white"
)

axes = np.atleast_1d(axes).flatten()

for i, alg_name in enumerate(valid_algs):
    ax = axes[i]
    x_hat = reconstructions[alg_name]
    n = min(len(t), len(x_hat))

    ax.plot(t[:n], x_ref[:n], label="Reference", linewidth=2)
    ax.plot(t[:n], x_hat[:n], label=alg_name, linewidth=1.5)

    if scenario in ["random_missing", "block_missing"]:
        missing_idx = np.where(mask[:n])[0]

        if len(missing_idx) > 0:
            ax.scatter(
                missing_idx,
                x_ref[:n][missing_idx],
                label="Missing locations",
                marker="x",
                s=20
            )

    ax.set_title(alg_name)
    ax.set_xlabel("Time index")
    ax.set_ylabel(parameter)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)

# Hide unused axes if necessary
for j in range(n_algs, len(axes)):
    axes[j].axis("off")

fig.suptitle(
    f"{parameter} reconstruction | {storm_label} | {scenario} | {severity}",
    fontsize=14,
    fontweight="bold"
)

plt.tight_layout(rect=[0, 0, 1, 0.97])

save_path = outdir / f"sample_reconstruction_grid_{scenario}_{parameter}_{storm_label}_{severity}.png"

plt.savefig(
    save_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print("Saved:", save_path)

## Figure S5

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from matplotlib.lines import Line2D
from pathlib import Path


# ============================================================
# SETTINGS
# ============================================================

# This file is used only to recover the ORIGINAL degradation
# realizations/seeds. It does not need to already contain
# the three new algorithms.
results_file = (
    "C:/Users/abadrud/2026-05-07 Update/"
    "results_fixed_rule/results_fixed_rule_raw_expanded.csv"
)

scenario = "block_missing"

output_dir = Path(
    "supplementary_block_missingness_analysis"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# DISPLAY ORDER
# ============================================================

parameter_order = [
    "Turbidity",
    "EC",
    "DO",
    "Temp"
]


phase_order = [
    "Baseline/inter-event",
    "Storm limb",
    "Peak window"
]


algorithm_order = [
    "Linear_Interpolation",
    "PCHIP",
    "Cubic_Spline",
    "Kalman_Filter",
    "SSA_Gap_Filling",
    "Local_Moving_Mean",
    "Delta_H_Weighted_Interpolation",
    "HPWI"
]


parameter_display = {
    "Turbidity": "Turbidity",
    "EC": "EC",
    "DO": "DO",
    "Temp": "Temperature"
}


algo_display = {
    "Linear_Interpolation": "Linear",
    "PCHIP": "PCHIP",
    "Cubic_Spline": "Spline",
    "Kalman_Filter": "Kalman",
    "SSA_Gap_Filling": "SSA gap",
    "Local_Moving_Mean": "Moving mean",
    "Delta_H_Weighted_Interpolation": "ΔH weighted",
    "HPWI": "HPWI"
}


parameter_aliases = {

    "Turbidity": [
        "Turbidity"
    ],

    "EC": [
        "EC",
        "Specific Conductance",
        "Specific_Conductance",
        "Conductivity"
    ],

    "DO": [
        "DO",
        "Dissolved Oxygen",
        "Dissolved_Oxygen"
    ],

    "Temp": [
        "Temp",
        "Temperature",
        "Water Temperature",
        "Water_Temperature"
    ]
}


# ============================================================
# CHECK REQUIRED OBJECTS
# ============================================================

required = [
    "all_storms",
    "classified_storms",
    "get_reference_signal",
    "get_storm_dataframe",
    "DEGRADATION_FUNCTIONS",
    "choose_algorithm_pool",
    "match_length",

    # New gap algorithms
    "local_moving_mean_gapfill",
    "delta_h_weighted_gapfill",
    "hpwi_gapfill"
]


missing = [
    name
    for name in required
    if name not in globals()
]


if missing:

    raise NameError(
        "Run the original experiment setup, hydrologic "
        "classification, and new-algorithm definition cells first.\n\n"
        "Missing:\n"
        + ", ".join(missing)
    )


# ============================================================
# HELPERS
# ============================================================

def storm_key(value):

    try:
        return int(float(value))

    except Exception:
        return value


def normalize_name(value):

    return (
        str(value)
        .strip()
        .lower()
        .replace("_", " ")
        .replace("-", " ")
    )


def find_parameter_column(
    df,
    parameter
):

    exact = {
        str(c): c
        for c in df.columns
    }


    normalized = {
        normalize_name(c): c
        for c in df.columns
    }


    for alias in parameter_aliases[
        parameter
    ]:

        if alias in exact:
            return exact[alias]

        key = normalize_name(
            alias
        )

        if key in normalized:
            return normalized[key]


    return None


# ============================================================
# FIND PHASE COLUMN
# ============================================================

def find_phase_column(df):

    candidates = [
        "Phase",
        "Hydrologic_Phase",
        "hydrologic_phase",
        "Hydrologic Phase",
        "storm_phase"
    ]


    normalized = {
        normalize_name(c): c
        for c in df.columns
    }


    for candidate in candidates:

        key = normalize_name(
            candidate
        )

        if key in normalized:

            return normalized[
                key
            ]


    raise ValueError(
        "Could not identify hydrologic phase column.\n"
        f"Available columns:\n{df.columns.tolist()}"
    )


# ============================================================
# FIND STAGE COLUMN
#
# Prefer Stage_smoothed because this is the transformed radar
# series already used in the hydrologic classification.
# ============================================================

def find_stage_column(df):

    preferred = [
        "Stage_smoothed",
        "Stage smoothed",
        "stage_smoothed",
        "Smoothed stage",
        "Stage_normalized",
        "Stage"
    ]


    normalized = {
        normalize_name(c): c
        for c in df.columns
    }


    for candidate in preferred:

        key = normalize_name(
            candidate
        )

        if key in normalized:

            column = normalized[
                key
            ]

            values = pd.to_numeric(
                df[column],
                errors="coerce"
            )

            if values.notna().sum() > 5:

                return column


    raise ValueError(
        "Could not identify Stage_smoothed column.\n"
        f"Available columns:\n{df.columns.tolist()}"
    )


# ============================================================
# NORMALIZE PHASE LABELS
# ============================================================

def standardize_phase(value):

    text = normalize_name(
        value
    )


    if "peak" in text:

        return "Peak window"


    if (
        "limb" in text
        or
        "active" in text
        or
        "storm" in text
    ):

        return "Storm limb"


    if (
        "baseline" in text
        or
        "inter event" in text
        or
        "interevent" in text
    ):

        return "Baseline/inter-event"


    return str(
        value
    )


# ============================================================
# NORMALIZE ALGORITHM RETURN VALUE
#
# Some versions of ΔH and HPWI return:
#
#     x_hat, diagnostics
#
# Others return only:
#
#     x_hat
#
# This keeps the reconstruction loop compatible with either.
# ============================================================

def extract_prediction(result):

    if isinstance(
        result,
        tuple
    ):

        return result[0]

    return result


# ============================================================
# LOAD ORIGINAL RESULTS AND RECOVER EXACT SEEDS
# ============================================================

raw = pd.read_csv(
    results_file
)


for col in [
    "parameter",
    "scenario",
    "severity",
    "algorithm",
    "evaluation_domain",
    "status"
]:

    if col in raw.columns:

        raw[col] = (
            raw[col]
            .astype(str)
            .str.strip()
        )


trials = raw[
    raw["scenario"] == scenario
].copy()


if "status" in trials.columns:

    ok = (
        trials["status"]
        .str.lower()
        .isin([
            "ok",
            "success",
            "successful",
            "completed"
        ])
    )

    if ok.any():

        trials = (
            trials[
                ok
            ]
            .copy()
        )


trial_cols = [
    "parameter",
    "storm_id",
    "severity",
    "repetition",
    "seed"
]


trials = (
    trials[
        trial_cols
    ]
    .dropna(
        subset=["seed"]
    )
    .drop_duplicates()
    .sort_values([
        "parameter",
        "storm_id",
        "severity",
        "repetition"
    ])
    .reset_index(
        drop=True
    )
)


trials["seed"] = (
    trials["seed"]
    .astype(int)
)


trials["repetition"] = (
    trials["repetition"]
    .astype(int)
)


trials["storm_key"] = (
    trials["storm_id"]
    .apply(storm_key)
)


print(
    "\n========================================"
)

print(
    "BLOCK-MISSINGNESS TRIALS"
)

print(
    "========================================"
)


print(
    trials
    .groupby("parameter")
    .size()
)


print(
    "\nTotal unique trials:",
    len(trials)
)


# Expected:
# 4 parameters × 4 storms × 2 severities × 30 repetitions
# = 960 trials

if len(trials) != 960:

    print(
        "\nWARNING: Expected 960 unique block-missingness "
        f"trials but found {len(trials)}."
    )


# ============================================================
# BUILD PHASE + STAGE LOOKUPS BEFORE RECONSTRUCTION
# ============================================================

phase_lookup = {}

stage_lookup = {}

alignment_report = []


unique_parameter_storms = (
    trials[
        [
            "parameter",
            "storm_key"
        ]
    ]
    .drop_duplicates()
)


print(
    "\nBuilding hydrologic phase and stage alignment..."
)


for _, row in unique_parameter_storms.iterrows():


    parameter = row[
        "parameter"
    ]


    storm_id = row[
        "storm_key"
    ]


    classified = classified_storms[
        storm_id
    ]


    storm_df = get_storm_dataframe(
        all_storms,
        storm_id
    )


    x_ref = np.asarray(
        get_reference_signal(
            all_storms=all_storms,
            storm_id=storm_id,
            parameter=parameter,
            dropna=True
        ),
        dtype=float
    )


    parameter_col = find_parameter_column(
        storm_df,
        parameter
    )


    if parameter_col is None:

        raise ValueError(
            f"Could not identify parameter column: "
            f"{parameter}, Storm {storm_id}"
        )


    wq_full = pd.to_numeric(
        storm_df[
            parameter_col
        ],
        errors="coerce"
    )


    valid_wq = (
        wq_full.notna()
    )


    wq = (
        wq_full[
            valid_wq
        ]
    )


    # --------------------------------------------------------
    # Verify exact WQ/reference alignment
    # --------------------------------------------------------

    if len(wq) != len(x_ref):

        raise ValueError(
            f"Length mismatch: {parameter}, Storm {storm_id}. "
            f"WQ={len(wq)}, reference={len(x_ref)}"
        )


    if not np.allclose(
        wq.to_numpy(
            dtype=float
        ),
        x_ref,
        rtol=1e-8,
        atol=1e-10
    ):

        raise ValueError(
            f"Reference-value mismatch: "
            f"{parameter}, Storm {storm_id}"
        )


    phase_col = find_phase_column(
        classified
    )


    stage_col = find_stage_column(
        classified
    )


    # --------------------------------------------------------
    # Preferred: index alignment
    # --------------------------------------------------------

    if wq.index.isin(
        classified.index
    ).all():


        phases = (
            classified[
                phase_col
            ]
            .reindex(
                wq.index
            )
        )


        stage = (
            pd.to_numeric(
                classified[
                    stage_col
                ],
                errors="coerce"
            )
            .reindex(
                wq.index
            )
        )


    # --------------------------------------------------------
    # Positional fallback
    # --------------------------------------------------------

    elif len(classified) == len(
        storm_df
    ):


        positions = np.flatnonzero(
            valid_wq.to_numpy()
        )


        phases = (
            classified[
                phase_col
            ]
            .iloc[
                positions
            ]
        )


        stage = (
            pd.to_numeric(
                classified[
                    stage_col
                ],
                errors="coerce"
            )
            .iloc[
                positions
            ]
        )


    else:

        raise ValueError(
            f"Cannot safely align hydrologic data: "
            f"{parameter}, Storm {storm_id}"
        )


    phases = np.array(
        [
            standardize_phase(p)
            for p in phases
        ],
        dtype=object
    )


    stage = np.asarray(
        stage,
        dtype=float
    )


    if len(phases) != len(x_ref):

        raise ValueError(
            f"Phase length mismatch: "
            f"{parameter}, Storm {storm_id}"
        )


    if len(stage) != len(x_ref):

        raise ValueError(
            f"Stage length mismatch: "
            f"{parameter}, Storm {storm_id}"
        )


    phase_lookup[
        (
            parameter,
            storm_id
        )
    ] = phases


    stage_lookup[
        (
            parameter,
            storm_id
        )
    ] = stage


    alignment_report.append({

        "parameter":
            parameter,

        "storm_id":
            storm_id,

        "phase_column":
            phase_col,

        "stage_column":
            stage_col,

        "n_reference":
            len(x_ref),

        "finite_stage_percent":
            100
            *
            np.isfinite(
                stage
            ).mean()
    })


alignment_report = pd.DataFrame(
    alignment_report
)


print(
    "\nHydrologic alignment report:"
)


print(
    alignment_report
    .sort_values([
        "parameter",
        "storm_id"
    ])
    .to_string(
        index=False
    )
)


# ============================================================
# VERIFY PHASE LABELS
# ============================================================

all_phase_labels = set()


for phases in phase_lookup.values():

    all_phase_labels.update(
        np.unique(phases)
    )


print(
    "\nHydrologic phases found:"
)

print(
    sorted(
        all_phase_labels
    )
)


unexpected = (
    all_phase_labels
    -
    set(
        phase_order
    )
)


if unexpected:

    raise ValueError(
        "Unexpected phase labels found: "
        f"{unexpected}"
    )


# ============================================================
# ORIGINAL ALGORITHM POOL
# ============================================================

original_pool = choose_algorithm_pool(
    scenario
)


print(
    "\nAlgorithms in original pool:"
)

print(
    list(
        original_pool.keys()
    )
)


# ============================================================
# RESOLVE ORIGINAL FIVE FUNCTIONS
# ============================================================

original_aliases = {

    "Linear_Interpolation": [
        "Linear_Interpolation",
        "Linear Interpolation",
        "Linear"
    ],

    "PCHIP": [
        "PCHIP"
    ],

    "Cubic_Spline": [
        "Cubic_Spline",
        "Cubic_Spline_Interpolation"
    ],

    "Kalman_Filter": [
        "Kalman_Filter",
        "Local_Level_Kalman_Smoother"
    ],

    "SSA_Gap_Filling": [
        "SSA_Gap_Filling"
    ]
}


def resolve_original_algorithm(
    canonical_name
):

    for candidate in original_aliases[
        canonical_name
    ]:

        if candidate in original_pool:

            return original_pool[
                candidate
            ]


    raise KeyError(
        f"Could not find {canonical_name} in original pool.\n"
        f"Available: {list(original_pool.keys())}"
    )


# ============================================================
# BUILD THE COMPLETE 8-ALGORITHM POOL
#
# IMPORTANT:
# stage changes for every parameter/storm combination, so this
# pool must be built INSIDE each degradation trial.
# ============================================================

def build_eight_algorithm_pool(
    scenario,
    stage
):

    # --------------------------------------------------------
    # Original five
    # --------------------------------------------------------

    pool = {

        "Linear_Interpolation":
            resolve_original_algorithm(
                "Linear_Interpolation"
            ),

        "PCHIP":
            resolve_original_algorithm(
                "PCHIP"
            ),

        "Cubic_Spline":
            resolve_original_algorithm(
                "Cubic_Spline"
            ),

        "Kalman_Filter":
            resolve_original_algorithm(
                "Kalman_Filter"
            ),

        "SSA_Gap_Filling":
            resolve_original_algorithm(
                "SSA_Gap_Filling"
            )
    }


    linear_fn = pool[
        "Linear_Interpolation"
    ]


    # --------------------------------------------------------
    # New 1 — Local Moving Mean
    # --------------------------------------------------------

    pool[
        "Local_Moving_Mean"
    ] = (

        lambda x:

            extract_prediction(

                local_moving_mean_gapfill(
                    x=x,
                    n_each_side=2
                )
            )
    )


    # --------------------------------------------------------
    # New 2 — ΔH Weighted
    # --------------------------------------------------------

    pool[
        "Delta_H_Weighted_Interpolation"
    ] = (

        lambda x,
        stage=stage,
        linear_fn=linear_fn:

            extract_prediction(

                delta_h_weighted_gapfill(
                    x=x,
                    stage=stage,
                    linear_function=linear_fn
                )
            )
    )


    # --------------------------------------------------------
    # New 3 — HPWI
    # --------------------------------------------------------

    pool[
        "HPWI"
    ] = (

        lambda x,
        stage=stage,
        linear_fn=linear_fn:

            extract_prediction(

                hpwi_gapfill(
                    x=x,
                    stage=stage,
                    linear_function=linear_fn
                )
            )
    )


    # --------------------------------------------------------
    # Force desired order
    # --------------------------------------------------------

    pool = {
        algorithm: pool[
            algorithm
        ]
        for algorithm in algorithm_order
    }


    if len(pool) != 8:

        raise RuntimeError(
            f"Expected 8 algorithms, found {len(pool)}."
        )


    return pool


# ============================================================
# REPRODUCE BLOCK-MISSINGNESS RECONSTRUCTIONS
# ALL 8 ALGORITHMS
# ============================================================

pointwise_rows = []


success_counter = {
    algorithm: 0
    for algorithm in algorithm_order
}


error_counter = {
    algorithm: 0
    for algorithm in algorithm_order
}


print(
    "\n========================================"
)

print(
    "RUNNING 8-ALGORITHM BLOCK ANALYSIS"
)

print(
    "========================================"
)


for count, trial in trials.iterrows():


    parameter = trial[
        "parameter"
    ]


    storm_id = trial[
        "storm_key"
    ]


    severity = trial[
        "severity"
    ]


    repetition = int(
        trial[
            "repetition"
        ]
    )


    seed = int(
        trial[
            "seed"
        ]
    )


    # --------------------------------------------------------
    # Reference
    # --------------------------------------------------------

    x_ref = np.asarray(
        get_reference_signal(
            all_storms=all_storms,
            storm_id=storm_id,
            parameter=parameter,
            dropna=True
        ),
        dtype=float
    )


    stage = stage_lookup[
        (
            parameter,
            storm_id
        )
    ]


    phases = phase_lookup[
        (
            parameter,
            storm_id
        )
    ]


    # --------------------------------------------------------
    # Exact original degradation
    # --------------------------------------------------------

    rng = np.random.default_rng(
        seed
    )


    x_deg, corruption_mask = (
        DEGRADATION_FUNCTIONS[
            scenario
        ](
            x_ref,
            severity,
            rng
        )
    )


    x_deg = np.asarray(
        x_deg,
        dtype=float
    )


    corruption_mask = np.asarray(
        corruption_mask,
        dtype=bool
    )


    # --------------------------------------------------------
    # Build all 8 using THIS storm's stage vector
    # --------------------------------------------------------

    algorithm_pool = (
        build_eight_algorithm_pool(
            scenario=scenario,
            stage=stage
        )
    )


    # ========================================================
    # RUN ALL EIGHT
    # ========================================================

    for algorithm, algorithm_fn in algorithm_pool.items():


        try:

            result = algorithm_fn(
                x_deg
            )


            x_hat = np.asarray(
                match_length(
                    extract_prediction(
                        result
                    ),
                    len(x_ref)
                ),
                dtype=float
            )


            # ------------------------------------------------
            # Only artificially missing points
            # ------------------------------------------------

            valid = (
                corruption_mask
                &
                np.isfinite(
                    x_ref
                )
                &
                np.isfinite(
                    x_hat
                )
            )


            missing_idx = np.where(
                valid
            )[0]


            if len(
                missing_idx
            ) == 0:

                continue


            pointwise_rows.append(

                pd.DataFrame({

                    "parameter":
                        parameter,

                    "storm_id":
                        storm_id,

                    "severity":
                        severity,

                    "repetition":
                        repetition,

                    "seed":
                        seed,

                    "algorithm":
                        algorithm,

                    "algorithm_display":
                        algo_display[
                            algorithm
                        ],

                    "point_index":
                        missing_idx,

                    "reference":
                        x_ref[
                            missing_idx
                        ],

                    "reconstructed":
                        x_hat[
                            missing_idx
                        ],

                    "Hydrologic_Phase":
                        phases[
                            missing_idx
                        ]
                })
            )


            success_counter[
                algorithm
            ] += 1


        except Exception as exc:


            error_counter[
                algorithm
            ] += 1


            # Only print first 5 errors / algorithm
            if error_counter[
                algorithm
            ] <= 5:

                print(
                    f"\nERROR — {algorithm}"
                    f"\nParameter: {parameter}"
                    f"\nStorm: {storm_id}"
                    f"\nSeverity: {severity}"
                    f"\nRepetition: {repetition}"
                    f"\nSeed: {seed}"
                    f"\nMessage: {exc}"
                )


    if (
        count + 1
    ) % 100 == 0:

        print(
            f"Processed "
            f"{count + 1}/{len(trials)} "
            f"degradation trials"
        )


# ============================================================
# EXECUTION CHECK
# ============================================================

print(
    "\n========================================"
)

print(
    "ALGORITHM EXECUTION SUMMARY"
)

print(
    "========================================"
)


for algorithm in algorithm_order:

    print(
        f"{algo_display[algorithm]:15s} "
        f"success={success_counter[algorithm]:4d} | "
        f"errors={error_counter[algorithm]:4d}"
    )


if not pointwise_rows:

    raise RuntimeError(
        "No pointwise results generated."
    )


pointwise = pd.concat(
    pointwise_rows,
    ignore_index=True
)


# ============================================================
# VERIFY ALL 8 ENTERED DATASET
# ============================================================

present_algorithms = (
    pointwise[
        "algorithm"
    ]
    .dropna()
    .unique()
    .tolist()
)


print(
    "\nAlgorithms actually present:"
)


for algorithm in algorithm_order:

    status = (
        "YES"
        if algorithm in present_algorithms
        else "NO"
    )

    print(
        f"{algo_display[algorithm]:15s}: "
        f"{status}"
    )


if len(
    present_algorithms
) != 8:

    raise RuntimeError(
        f"Expected 8 algorithms but found "
        f"{len(present_algorithms)}."
    )


print(
    "\nPASS: All 8 algorithms are present."
)


# ============================================================
# ERROR VARIABLES
# ============================================================

pointwise[
    "Error"
] = (
    pointwise[
        "reconstructed"
    ]
    -
    pointwise[
        "reference"
    ]
)


pointwise[
    "Absolute_Error"
] = np.abs(
    pointwise[
        "Error"
    ]
)


pointwise[
    "Squared_Error"
] = (
    pointwise[
        "Error"
    ]
    ** 2
)


# ============================================================
# SAVE COMPLETE POINTWISE FILE
# ============================================================

pointwise.to_csv(

    output_dir
    /
    "block_missingness_pointwise_with_hydrologic_phase.csv",

    index=False
)


print(
    "\nTotal algorithm-specific reconstructed "
    "missing-point observations:"
)

print(
    f"{len(pointwise):,}"
)


# ============================================================
# UNIQUE MISSING OBSERVATIONS
#
# Remove 8-fold algorithm duplication.
# ============================================================

missing_keys = [
    "parameter",
    "storm_id",
    "severity",
    "repetition",
    "seed",
    "point_index"
]


unique_missing = (
    pointwise
    .drop_duplicates(
        missing_keys
    )
)


coverage = (
    len(pointwise)
    /
    len(unique_missing)
)


print(
    "\nUnique artificially missing observations:",
    f"{len(unique_missing):,}"
)


print(
    "Average algorithm coverage per missing point:",
    f"{coverage:.2f}"
)


# Should be 8.00
if not np.isclose(
    coverage,
    8.0,
    atol=0.01
):

    print(
        "\nWARNING: average coverage is not exactly 8.00."
    )


# ============================================================
# DISTRIBUTION BY HYDROLOGIC PHASE
# ============================================================

phase_distribution = (

    unique_missing

    .groupby(
        [
            "parameter",
            "Hydrologic_Phase"
        ]
    )

    .size()

    .rename(
        "n_missing"
    )

    .reset_index()
)


phase_distribution[
    "Percent"
] = (

    phase_distribution[
        "n_missing"
    ]

    /

    phase_distribution
    .groupby(
        "parameter"
    )[
        "n_missing"
    ]
    .transform(
        "sum"
    )

    *
    100
)


phase_distribution.to_csv(

    output_dir
    /
    "block_missing_points_by_hydrologic_phase.csv",

    index=False
)


print(
    "\nMissing-point distribution by hydrologic phase (%):"
)


print(

    phase_distribution

    .pivot(
        index=
            "Hydrologic_Phase",
        columns=
            "parameter",
        values=
            "Percent"
    )

    .reindex(
        phase_order
    )

    .round(
        1
    )
)


# ============================================================
# TRIAL-LEVEL PHASE METRICS
#
# Experimental unit:
# parameter × storm × severity × repetition × seed
# × algorithm × hydrologic phase
# ============================================================

trial_group = [
    "parameter",
    "storm_id",
    "severity",
    "repetition",
    "seed",
    "algorithm",
    "Hydrologic_Phase"
]


trial_phase = (

    pointwise

    .groupby(
        trial_group,
        as_index=False
    )

    .agg(

        n_missing=(
            "Absolute_Error",
            "size"
        ),

        MAE=(
            "Absolute_Error",
            "mean"
        ),

        MSE=(
            "Squared_Error",
            "mean"
        ),

        Bias=(
            "Error",
            "mean"
        )
    )
)


trial_phase[
    "RMSE"
] = np.sqrt(
    trial_phase[
        "MSE"
    ]
)


trial_phase.drop(
    columns="MSE",
    inplace=True
)


# ============================================================
# RANK WITHIN MATCHED TRIAL + PHASE
# ============================================================

rank_group = [
    "parameter",
    "storm_id",
    "severity",
    "repetition",
    "seed",
    "Hydrologic_Phase"
]


trial_phase[
    "MAE_Rank"
] = (

    trial_phase

    .groupby(
        rank_group
    )[
        "MAE"
    ]

    .rank(
        method="average",
        ascending=True
    )
)


trial_phase[
    "RMSE_Rank"
] = (

    trial_phase

    .groupby(
        rank_group
    )[
        "RMSE"
    ]

    .rank(
        method="average",
        ascending=True
    )
)


# ============================================================
# VERIFY 8 ALGORITHMS PER MATCHED PHASE TRIAL
# ============================================================

group_sizes = (

    trial_phase

    .groupby(
        rank_group
    )[
        "algorithm"
    ]

    .nunique()
)


print(
    "\nAlgorithms available per matched "
    "phase-specific trial:"
)


print(
    group_sizes
    .value_counts()
    .sort_index()
)


if group_sizes.min() != 8:

    print(
        "\nWARNING: At least one matched trial/phase "
        "contains fewer than 8 algorithms."
    )


trial_phase.to_csv(

    output_dir
    /
    "block_trial_level_performance_by_hydrologic_phase.csv",

    index=False
)


# ============================================================
# SUMMARY ACROSS TRIALS
#
# This file feeds Table S7.
# ============================================================

phase_summary = (

    trial_phase

    .groupby(
        [
            "parameter",
            "algorithm",
            "Hydrologic_Phase"
        ],
        as_index=False
    )

    .agg(

        n_trials=(
            "MAE",
            "size"
        ),

        Mean_MAE=(
            "MAE",
            "mean"
        ),

        SD_MAE=(
            "MAE",
            "std"
        ),

        Median_MAE=(
            "MAE",
            "median"
        ),

        Mean_RMSE=(
            "RMSE",
            "mean"
        ),

        SD_RMSE=(
            "RMSE",
            "std"
        ),

        Median_RMSE=(
            "RMSE",
            "median"
        ),

        Mean_MAE_Rank=(
            "MAE_Rank",
            "mean"
        ),

        Mean_RMSE_Rank=(
            "RMSE_Rank",
            "mean"
        )
    )
)


phase_summary.to_csv(

    output_dir
    /
    "block_performance_summary_by_hydrologic_phase.csv",

    index=False
)


# ============================================================
# SEVERITY-SPECIFIC SUMMARY
# ============================================================

phase_summary_severity = (

    trial_phase

    .groupby(
        [
            "parameter",
            "severity",
            "algorithm",
            "Hydrologic_Phase"
        ],
        as_index=False
    )

    .agg(

        n_trials=(
            "MAE",
            "size"
        ),

        Mean_MAE=(
            "MAE",
            "mean"
        ),

        SD_MAE=(
            "MAE",
            "std"
        ),

        Mean_RMSE=(
            "RMSE",
            "mean"
        ),

        SD_RMSE=(
            "RMSE",
            "std"
        ),

        Mean_MAE_Rank=(
            "MAE_Rank",
            "mean"
        ),

        Mean_RMSE_Rank=(
            "RMSE_Rank",
            "mean"
        )
    )
)


phase_summary_severity.to_csv(

    output_dir
    /
    "block_performance_by_phase_and_severity.csv",

    index=False
)


# ============================================================
# PRINT PRIMARY RESULTS
# ============================================================

print(
    "\n========================================"
)

print(
    "MEAN TRIAL-LEVEL MAE BY PHASE"
)

print(
    "========================================"
)


print(

    phase_summary

    .pivot_table(
        index=[
            "parameter",
            "algorithm"
        ],
        columns=
            "Hydrologic_Phase",
        values=
            "Mean_MAE"
    )

    .reindex(
        columns=
            phase_order
    )

    .round(
        3
    )
)


print(
    "\n========================================"
)

print(
    "MEAN TRIAL-LEVEL RMSE BY PHASE"
)

print(
    "========================================"
)


print(

    phase_summary

    .pivot_table(
        index=[
            "parameter",
            "algorithm"
        ],
        columns=
            "Hydrologic_Phase",
        values=
            "Mean_RMSE"
    )

    .reindex(
        columns=
            phase_order
    )

    .round(
        3
    )
)


print(
    "\n========================================"
)

print(
    "MEAN MAE RANK BY PHASE"
)

print(
    "========================================"
)


print(

    phase_summary

    .pivot_table(
        index=[
            "parameter",
            "algorithm"
        ],
        columns=
            "Hydrologic_Phase",
        values=
            "Mean_MAE_Rank"
    )

    .reindex(
        columns=
            phase_order
    )

    .round(
        2
    )
)


# ============================================================
# SUBSAMPLE FOR PLOTTING ONLY
#
# All statistics above use every reconstructed point.
# ============================================================

MAX_POINTS_PER_PHASE = 1500


plot_parts = []


for _, group in pointwise.groupby(
    [
        "parameter",
        "algorithm",
        "Hydrologic_Phase"
    ]
):

    n = min(
        len(group),
        MAX_POINTS_PER_PHASE
    )


    plot_parts.append(

        group.sample(
            n=n,
            random_state=2026
        )
    )


plot_data = pd.concat(
    plot_parts,
    ignore_index=True
)


# ============================================================
# PLOT SETTINGS
# ============================================================

phase_colors = {
    "Baseline/inter-event": "#1f77b4",
    "Storm limb": "#ff7f0e",
    "Peak window": "#2ca02c"
}


plt.rcParams[
    "font.family"
] = "serif"


plt.rcParams[
    "font.serif"
] = [
    "Times New Roman",
    "Times",
    "DejaVu Serif"
]


# ============================================================
# ROBUST COMMON LIMITS BY PARAMETER
#
# Limits use all 8 algorithms and are then reused in BOTH
# 4 × 4 figures.
#
# Negative reconstructions remain in statistics.
# Turbidity display begins at 0.
# ============================================================

parameter_limits = {}


for parameter in parameter_order:


    p_all = pointwise[
        pointwise[
            "parameter"
        ]
        ==
        parameter
    ]


    values = np.concatenate([
        p_all[
            "reference"
        ].to_numpy(),

        p_all[
            "reconstructed"
        ].to_numpy()
    ])


    values = values[
        np.isfinite(
            values
        )
    ]


    lo = np.nanpercentile(
        values,
        0.2
    )


    hi = np.nanpercentile(
        values,
        99.8
    )


    pad = (
        0.04
        *
        (
            hi - lo
        )
        if hi > lo
        else 1
    )


    lo = lo - pad

    hi = hi + pad


    # Display boundary only
    if parameter == "Turbidity":

        lo = 0.0


    parameter_limits[
        parameter
    ] = (
        lo,
        hi
    )


print(
    "\nParameter plotting limits:"
)


for parameter in parameter_order:

    print(
        parameter,
        parameter_limits[
            parameter
        ]
    )


# ============================================================
# SPLIT INTO TWO 4 × 4 FIGURES
# ============================================================

figure_groups = {

    "A": [
        "Linear_Interpolation",
        "PCHIP",
        "Cubic_Spline",
        "Local_Moving_Mean"
    ],

    "B": [
        "Delta_H_Weighted_Interpolation",
        "HPWI",
        "Kalman_Filter",
        "SSA_Gap_Filling"
    ]
}


# ============================================================
# LEGEND
# ============================================================

handles = [

    Line2D(
        [0],
        [0],
        marker="o",
        linestyle="",
        markersize=7,
        markerfacecolor=
            phase_colors[
                p
            ],
        markeredgecolor="none",
        label=p
    )

    for p in phase_order
]


handles += [

    Line2D(
        [0],
        [0],
        linestyle="--",
        color="black",
        linewidth=1.4,
        label="1:1 line"
    ),

    Line2D(
        [0],
        [0],
        linestyle="-",
        color="0.45",
        linewidth=1.2,
        label="Regression fit"
    )
]


# ============================================================
# CREATE BOTH FIGURES
# ============================================================

for figure_name, group_algorithms in figure_groups.items():


    fig, axes = plt.subplots(
        4,
        4,
        figsize=(
            13.5,
            12
        ),
        facecolor=
            "white"
    )


    for row, parameter in enumerate(
        parameter_order
    ):


        lo, hi = parameter_limits[
            parameter
        ]


        for col, algorithm in enumerate(
            group_algorithms
        ):


            ax = axes[
                row,
                col
            ]


            data = plot_data[
                (
                    plot_data[
                        "parameter"
                    ]
                    ==
                    parameter
                )
                &
                (
                    plot_data[
                        "algorithm"
                    ]
                    ==
                    algorithm
                )
            ]


            # ------------------------------------------------
            # Scatter by phase
            # ------------------------------------------------

            for phase in phase_order:


                d = data[
                    data[
                        "Hydrologic_Phase"
                    ]
                    ==
                    phase
                ]


                ax.scatter(
                    d[
                        "reference"
                    ],
                    d[
                        "reconstructed"
                    ],
                    s=8,
                    alpha=0.30,
                    color=
                        phase_colors[
                            phase
                        ],
                    edgecolors="none",
                    rasterized=True
                )


            # ------------------------------------------------
            # Regression line from ALL points
            # ------------------------------------------------

            full = pointwise[
                (
                    pointwise[
                        "parameter"
                    ]
                    ==
                    parameter
                )
                &
                (
                    pointwise[
                        "algorithm"
                    ]
                    ==
                    algorithm
                )
            ]


            x = full[
                "reference"
            ].to_numpy(
                dtype=float
            )


            y = full[
                "reconstructed"
            ].to_numpy(
                dtype=float
            )


            valid = (
                np.isfinite(x)
                &
                np.isfinite(y)
            )


            if valid.sum() > 2:


                slope, intercept = np.polyfit(
                    x[
                        valid
                    ],
                    y[
                        valid
                    ],
                    1
                )


                xx = np.linspace(
                    lo,
                    hi,
                    100
                )


                ax.plot(
                    xx,
                    slope
                    *
                    xx
                    +
                    intercept,
                    color="0.45",
                    linewidth=1.2,
                    zorder=4
                )


            # ------------------------------------------------
            # 1:1 line
            # ------------------------------------------------

            ax.plot(
                [
                    lo,
                    hi
                ],
                [
                    lo,
                    hi
                ],
                "--",
                color="black",
                linewidth=1.4,
                zorder=5
            )


            ax.set_xlim(
                lo,
                hi
            )


            ax.set_ylim(
                lo,
                hi
            )


            if row == 0:

                ax.set_title(
                    algo_display[
                        algorithm
                    ],
                    fontsize=13,
                    fontweight="bold"
                )


            if col == 0:

                ax.set_ylabel(
                    f"{parameter_display[parameter]}\nReconstructed",
                    fontsize=11
                )


            if row == len(
                parameter_order
            ) - 1:

                ax.set_xlabel(
                    "Reference",
                    fontsize=11
                )


            ax.tick_params(
                labelsize=8
            )


    # --------------------------------------------------------
    # Shared legend
    # --------------------------------------------------------

    fig.legend(
        handles=handles,
        loc="lower center",
        ncol=5,
        frameon=False,
        fontsize=10,
        bbox_to_anchor=(
            0.5,
            0.01
        )
    )


    fig.subplots_adjust(
        left=0.09,
        right=0.99,
        top=0.95,
        bottom=0.09,
        wspace=0.22,
        hspace=0.28
    )


    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    png_path = (
        output_dir
        /
        f"block_missingness_parity_by_hydrologic_phase_"
        f"part_{figure_name}.png"
    )


    pdf_path = (
        output_dir
        /
        f"block_missingness_parity_by_hydrologic_phase_"
        f"part_{figure_name}.pdf"
    )


    fig.savefig(
        png_path,
        dpi=600,
        bbox_inches="tight",
        facecolor="white"
    )


    fig.savefig(
        pdf_path,
        bbox_inches="tight",
        facecolor="white"
    )


    plt.show()


# ============================================================
# FINAL
# ============================================================

print(
    "\n========================================"
)

print(
    "BLOCK-MISSINGNESS PHASE ANALYSIS COMPLETE"
)

print(
    "========================================"
)


print(
    "\nAlgorithms analyzed:"
)


for algorithm in algorithm_order:

    print(
        " -",
        algo_display[
            algorithm
        ]
    )


print(
    "\nNumber of algorithms:",
    pointwise[
        "algorithm"
    ].nunique()
)


print(
    "\nOutputs saved to:"
)

print(
    output_dir.resolve()
)

## Table S7 and S8


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

outdir = Path("analysis_outputs")
outdir.mkdir(exist_ok=True)

scenario_name = "additive_noise"   # change to "random_perturbation" if that is your scenario name

# Use your clean summary dataframe
df = raw[
    (raw["status"] == "ok") &
    (raw["scenario"] == scenario_name) &
    (raw["evaluation_domain"].isin(["full", "full_series"]))
].copy()

# ---------------------------------------------------------
# Multi-metric table by parameter and algorithm
# ---------------------------------------------------------
metric_by_parameter = (
    df
    .groupby(["parameter", "algorithm"], as_index=False)
    .agg(
        RMSE=("RMSE", "mean"),
        RMSE_sd=("RMSE", "std"),
        
        MAE=("MAE", "mean"),
        MAE_sd=("MAE", "std"),

        KGE=("KGE", "mean"),
        KGE_sd=("KGE", "std"),

        NSE=("NSE", "mean"),
        NSE_sd=("NSE", "std"),

        Pearson_r=("Pearson_r", "mean"),
        Pearson_r_sd=("Pearson_r", "std"),

        Peak_Error=("Peak_Error", "mean"),
        Peak_Error_sd=("Peak_Error", "std"),

        Variance_Ratio=("Variance_Ratio", "mean"),
        Variance_Ratio_sd=("Variance_Ratio", "std"),

        Shannon_Entropy_Diff=("Shannon_Entropy_Diff", "mean"),
        Shannon_Entropy_Diff_sd=("Shannon_Entropy_Diff", "std"),
    )
)

# ---------------------------------------------------------
# Add parameter-specific ranks
# ---------------------------------------------------------
metric_by_parameter["RMSE_rank"] = metric_by_parameter.groupby("parameter")["RMSE"].rank(
    ascending=True, method="min"
)

metric_by_parameter["MAE_rank"] = metric_by_parameter.groupby("parameter")["MAE"].rank(
    ascending=True, method="min"
)

metric_by_parameter["KGE_rank"] = metric_by_parameter.groupby("parameter")["KGE"].rank(
    ascending=False, method="min"
)

metric_by_parameter["NSE_rank"] = metric_by_parameter.groupby("parameter")["NSE"].rank(
    ascending=False, method="min"
)

metric_by_parameter["Pearson_rank"] = metric_by_parameter.groupby("parameter")["Pearson_r"].rank(
    ascending=False, method="min"
)

metric_by_parameter["Peak_rank"] = metric_by_parameter.groupby("parameter")["Peak_Error"].apply(
    lambda x: x.abs().rank(ascending=True, method="min")
).reset_index(level=0, drop=True)

metric_by_parameter["Variance_rank"] = metric_by_parameter.groupby("parameter")["Variance_Ratio"].apply(
    lambda x: (x - 1).abs().rank(ascending=True, method="min")
).reset_index(level=0, drop=True)

metric_by_parameter["Entropy_rank"] = metric_by_parameter.groupby("parameter")["Shannon_Entropy_Diff"].rank(
    ascending=True, method="min"
)

rank_cols = [
    "RMSE_rank",
    "MAE_rank",
    "KGE_rank",
    "NSE_rank",
    "Pearson_rank",
    "Peak_rank",
    "Variance_rank",
    "Entropy_rank",
]

metric_by_parameter["Mean_Metric_Rank"] = metric_by_parameter[rank_cols].mean(axis=1)

metric_by_parameter = metric_by_parameter.sort_values(
    ["parameter", "Mean_Metric_Rank"]
)

metric_by_parameter_round = metric_by_parameter.copy()
numeric_cols = metric_by_parameter_round.select_dtypes(include="number").columns
metric_by_parameter_round[numeric_cols] = metric_by_parameter_round[numeric_cols].round(4)

display(metric_by_parameter_round)

metric_by_parameter_round.to_csv(
    outdir / f"metric_values_and_ranks_by_parameter_{scenario_name}.csv",
    index=False
)

In [ ]:
formatted = metric_by_parameter.copy()

def mean_sd(mean_col, sd_col, digits=3):
    return (
        formatted[mean_col].round(digits).astype(str)
        + " ± "
        + formatted[sd_col].round(digits).astype(str)
    )

formatted["RMSE"] = mean_sd("RMSE", "RMSE_sd")
formatted["MAE"] = mean_sd("MAE", "MAE_sd")
formatted["KGE"] = mean_sd("KGE", "KGE_sd")
formatted["NSE"] = mean_sd("NSE", "NSE_sd")
formatted["Pearson r"] = mean_sd("Pearson_r", "Pearson_r_sd")
formatted["Peak Error"] = mean_sd("Peak_Error", "Peak_Error_sd")
formatted["VR"] = mean_sd("Variance_Ratio", "Variance_Ratio_sd")
formatted["SE Diff"] = mean_sd("Shannon_Entropy_Diff", "Shannon_Entropy_Diff_sd")

formatted["Mean Metric Rank"] = formatted["Mean_Metric_Rank"].round(2)

formatted_table = formatted[
    [
        "parameter",
        "algorithm",
        "RMSE",
        "MAE",
        "KGE",
        "NSE",
        "Pearson r",
        "Peak Error",
        "VR",
        "SE Diff",
        "Mean Metric Rank",
    ]
].rename(
    columns={
        "parameter": "Parameter",
        "algorithm": "Algorithm",
    }
)

display(formatted_table)

formatted_table.to_csv(
    outdir / f"manuscript_metric_table_by_parameter_{scenario_name}.csv",
    index=False
)

## Table S9

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ----------------------------------------------------
# File paths
# ----------------------------------------------------
summary_file = Path("frequency_sensitivity_summary.csv")

output_dir = Path("frequency_sensitivity_outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------
# Load data
# ----------------------------------------------------
summary = pd.read_csv(summary_file)

print(summary.shape)
print(summary.columns.tolist())
print(summary[["sampling_frequency", "scenario", "parameter", "algorithm", "evaluation_domain"]].head())

In [ ]:
# ----------------------------------------------------
# Scenario-specific evaluation domain
# ----------------------------------------------------
def preferred_domain(scenario):
    """
    Select the correct evaluation domain for each degradation scenario.
    """
    if scenario in ["random_missing", "block_missing"]:
        return "corrupted_only"
    elif scenario in ["additive_noise", "field_realistic_fault"]:
        return "full"
    else:
        return "full"


summary["preferred_domain"] = summary["scenario"].apply(preferred_domain)

summary_main = summary[
    summary["evaluation_domain"] == summary["preferred_domain"]
].copy()

print(summary_main.shape)
print(summary_main.groupby(["scenario", "evaluation_domain"]).size())

In [ ]:
# Save as Excel-friendly CSV
table_for_paper.to_csv(
    output_dir / "Table_X_best_algorithm_by_frequency_for_manuscript.csv",
    index=False
)

print("Saved table:")
print(output_dir / "Table_X_best_algorithm_by_frequency_for_manuscript.csv")